# Demostrador en Google Colab (cuaderno autocontenido)

Un modelo local de **Ollama** *propone* y un verificador externo *decide*: **Lean 4** para matemáticas y **SymPy** para física.
Guarda el progreso tras **cada intento**: si lo interrumpes (botón de parar, desconexión, límite de tiempo) o falla,
**volver a ejecutar la celda continúa donde se quedó**.

**El código y los problemas de ejemplo están dentro de este cuaderno** (celda *Código del demostrador*): no hace falta `git clone` ni ningún otro archivo. Si lo cambias, vuelve a ejecutar esa celda.

Menú *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.

## 1. Instalar Ollama, Lean 4 y el modelo (una vez por sesión)

In [ ]:
# Dependencias y Ollama
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install sympy

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(6)

In [ ]:
# Lean 4 (vía elan)
!curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y --default-toolchain stable
!/root/.elan/bin/lean --version

In [ ]:
MODELO = 'qwen2.5:7b'
!ollama pull {MODELO}

## 2. Dónde guardar el progreso

Con Google Drive montado, el progreso **sobrevive** aunque se cierre la sesión.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/demostrador'      # estado y resultados
LEAN = '/root/.elan/bin/lean'

## 3. Código del demostrador

Ejecuta esta celda (y la siguiente) antes de resolver. Contiene el programa completo.

In [ ]:
%%writefile demostrador.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Demostrador - resuelve problemas matemáticos y físicos con un LLM local (Ollama) y un verificador externo.

Un solo archivo, solo biblioteca estándar. El modelo PROPONE; el verificador DECIDE:
  * backend "lean"  : demostraciones formales en Lean 4 (matemáticas). Lean es el juez.
  * backend "sympy" : derivaciones físicas/matemáticas con SymPy; se comprueban criterios fijados antes de resolver.

Pensado para problemas LARGOS y para poder INTERRUMPIRLO:
  * todo el progreso (árbol de demostración, intentos, errores) se guarda tras CADA intento (escritura atómica);
  * al volver a ejecutar el mismo comando continúa exactamente donde se quedó;
  * si un objetivo no sale, lo descompone en lemas (recursivamente) y conserva los que ya demostró;
  * presupuesto de tiempo/intentos (--horas, --max-intentos) y parada limpia con Ctrl-C / SIGTERM.

Uso:
  python demostrador.py resolver --problema par_mayor
  python demostrador.py resolver --enunciado "Demuestra que ..." --backend lean --dir mi_ejecucion
  python demostrador.py problemas                      (lista los problemas incluidos)
  python demostrador.py estado   --dir mi_ejecucion
  python demostrador.py selftest
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import textwrap
import time
import traceback
import unicodedata
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

VERSION = "4.0.0"
ESTADO_VERSION = 6
MODELO_DEFECTO = "qwen2.5:7b"
AXIOMAS_PERMITIDOS = {"propext", "Classical.choice", "Quot.sound"}


# =============================================================================
# 1. CONFIGURACIÓN
# =============================================================================
class Config:
    def __init__(self, **kw: Any) -> None:
        self.modelo = MODELO_DEFECTO
        self.ollama_url = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
        self.num_ctx = 8192
        self.num_predict = 1000
        self.backend = "lean"
        self.lean = ""              # ruta a lean (vacío = buscar en PATH y ~/.elan/bin)
        self.lake_dir = ""          # proyecto lake con Mathlib: se usa `lake env lean` dentro de él
        self.mathlib = False
        self.timeout_lean = 90
        self.timeout_py = 120
        self.formalizar = 8         # intentos de formalizar el enunciado
        self.directos = 6           # intentos directos por objetivo
        self.finales = 6            # intentos del objetivo tras demostrar sus lemas
        self.planes = 4             # intentos de obtener un plan válido
        self.replanes = 3           # replanificaciones por objetivo
        self.max_lemas = 4
        self.profundidad = 2        # niveles de descomposición recursiva
        self.fidelidad = True       # comprobar que el enunciado formal dice lo que pide el problema
        self.horas = 0.0            # 0 = sin límite de tiempo
        self.max_intentos = 0       # 0 = sin límite de llamadas al modelo
        self.semilla = 1
        self.biblioteca = ""        # JSON con demostraciones verificadas previas (aprendizaje sin tocar pesos)
        for k, v in kw.items():
            if not hasattr(self, k):
                raise TypeError(f"opción desconocida: {k}")
            setattr(self, k, v)


class Pausa(Exception):
    """Se alcanzó el presupuesto (tiempo/intentos) o se pidió parar: el estado ya está guardado."""


class ErrorModelo(Exception):
    pass


def ahora() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


# =============================================================================
# 2. UTILIDADES
# =============================================================================
def ensamblar(*partes: Any) -> str:
    return "\n\n".join(p.strip() for p in partes if isinstance(p, str) and p.strip())


def normalizar(texto: str) -> str:
    # Sin NFKC a propósito: NFKC convertiría ℕ en "N" antes de reconocerlo, y la versión C++ no dispone de él;
    # así Python y C++ calculan la misma huella y pueden compartir el estado guardado.
    t = (texto or "").replace("ℕ", "Nat").replace("ℤ", "Int").replace("ℝ", "Real").replace("→", "->")
    t = t.replace("≤", "<=").replace("≥", ">=")
    return re.sub(r"\s+", " ", t).strip()


def huella(texto: str) -> str:
    return hashlib.sha256(normalizar(texto).encode("utf-8")).hexdigest()[:16]


def recortar(texto: str, n: int) -> str:
    texto = str(texto)
    return texto if len(texto) <= n else texto[: n - 15] + f"... [+{len(texto) - n + 15}]"


def quitar_comentarios_lean(codigo: str) -> str:
    codigo = re.sub(r"/-.*?-/", "", codigo, flags=re.S)
    return re.sub(r"--.*", "", codigo)


def quitar_comentarios_py(codigo: str) -> str:
    return re.sub(r"#.*", "", codigo)


def escribir_atomico(ruta: Path, texto: str) -> None:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tmp = ruta.with_name(ruta.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(texto)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    os.replace(tmp, ruta)


def extraer_json(texto: str) -> Dict[str, Any]:
    t = (texto or "").strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    try:
        obj = json.loads(t)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass
    i = t.find("{")
    if i < 0:
        raise ValueError("el modelo no devolvió JSON")
    prof, instr, esc = 0, False, False
    for k in range(i, len(t)):
        c = t[k]
        if instr:
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                instr = False
        elif c == '"':
            instr = True
        elif c == "{":
            prof += 1
        elif c == "}":
            prof -= 1
            if prof == 0:
                obj = json.loads(t[i:k + 1])
                if isinstance(obj, dict):
                    return obj
                break
    raise ValueError("el modelo no devolvió un objeto JSON válido")


def bloques_codigo(texto: str, lenguajes: Tuple[str, ...]) -> List[str]:
    pat = r"```(?:" + "|".join(lenguajes) + r")?[ \t]*\n(.*?)```"
    return [b.strip("\n") for b in re.findall(pat, texto or "", flags=re.S | re.I)]


def elegir_bloque(texto: str, lenguajes: Tuple[str, ...], pista: Callable[[str], bool]) -> str:
    """De varias vallas de código, la que cumple `pista` (la última que la cumple); si no, la más larga."""
    bs = bloques_codigo(texto, lenguajes)
    if not bs:
        return (texto or "").replace("```", "").strip("\n")
    buenos = [b for b in bs if pista(b)]
    return buenos[-1] if buenos else max(bs, key=len)


def sangrar(lineas: List[str], n: int = 2) -> str:
    return "\n".join((" " * n + l) if l.strip() else l for l in lineas)


# =============================================================================
# 3. CLIENTE DE OLLAMA
# =============================================================================
class LLM:
    """Cliente mínimo de la API de Ollama con reintentos. Nunca usa proxy (el servidor es local)."""

    def __init__(self, cfg: Config) -> None:
        raw = (cfg.ollama_url or "http://127.0.0.1:11434").strip()
        if "://" not in raw:
            raw = "http://" + raw
        self.base = raw.replace("://0.0.0.0", "://127.0.0.1").rstrip("/")
        self.modelo = cfg.modelo
        self.num_ctx = cfg.num_ctx
        self._op = urllib.request.build_opener(urllib.request.ProxyHandler({}))

    def _post(self, ruta: str, payload: Dict[str, Any], timeout: float = 1800.0, reintentos: int = 3) -> Dict[str, Any]:
        req = urllib.request.Request(self.base + ruta, data=json.dumps(payload).encode("utf-8"), method="POST",
                                     headers={"Content-Type": "application/json"})
        ultimo: Optional[BaseException] = None
        for i in range(reintentos + 1):
            try:
                with self._op.open(req, timeout=timeout) as r:
                    return json.loads(r.read().decode("utf-8"))
            except urllib.error.HTTPError as e:
                raise ErrorModelo(f"Ollama respondió HTTP {e.code}: {e.read().decode('utf-8', 'ignore')[:300]}")
            except (urllib.error.URLError, ConnectionError, socket.timeout, TimeoutError) as e:
                ultimo = e
                time.sleep(min(30, 2 * (i + 1)))
        raise ErrorModelo(f"No se puede hablar con Ollama en {self.base} ({ultimo}). ¿Está en marcha? (ollama serve)")

    def chat(self, sistema: str, usuario: str, *, temperatura: float = 0.2, num_predict: int = 1600, seed: int = 0,
             json_mode: bool = False) -> str:
        payload: Dict[str, Any] = {
            "model": self.modelo, "stream": False, "keep_alive": "60m",
            "messages": [{"role": "system", "content": sistema}, {"role": "user", "content": usuario}],
            "options": {"temperature": temperatura, "num_predict": num_predict, "num_ctx": self.num_ctx, "seed": seed},
        }
        if json_mode:
            payload["format"] = "json"
        out = self._post("/api/chat", payload)
        return str((out.get("message") or {}).get("content", ""))

    def modelos(self) -> List[str]:
        req = urllib.request.Request(self.base + "/api/tags")
        with self._op.open(req, timeout=10) as r:
            return [m.get("name", "") for m in json.loads(r.read().decode("utf-8")).get("models", [])]


# =============================================================================
# 4. VERIFICADORES
# =============================================================================
class Verificador:
    """Interfaz de un verificador. El motor no sabe si por debajo hay Lean o SymPy."""
    nombre = ""
    extension = ""
    admite_plan = False
    sistema = ""

    def entorno(self) -> str:
        raise NotImplementedError

    def comprobar(self, codigo: str, nombre_final: str = "") -> Tuple[bool, str]:
        raise NotImplementedError

    def prohibido(self, texto: str) -> Optional[str]:
        return None

    def titulo_error(self, resumen: str) -> str:
        """Línea más informativa de un error, para el registro de progreso."""
        lineas = [l for l in (resumen or "").splitlines() if l.strip()]
        return lineas[0] if lineas else ""


# ------------------------------------------------------------------ Lean 4
LEAN_PROHIBIDO = re.compile(
    r"\b(sorry|admit|axiom|native_decide|unsafe|implemented_by|extern|opaque|macro|macro_rules|syntax|elab|"
    r"notation|infix|infixl|infixr|prefix|postfix|set_option|run_cmd|initialize|builtin_initialize|trace|dbg_trace|dbgTrace|ofReduceBool|bv_decide|namespace|attribute)\b|"
    r"\+native|#eval|#exit|#reduce|^\s*import\b",
    re.M)
LEAN_ORDENES = re.compile(r"^(theorem|lemma|example|def|abbrev|instance|structure|inductive|class|namespace|end|open|section|"
                          r"variable|universe|attribute|noncomputable|private|protected|#)\b")

PISTA_SIN_MATHLIB = ("PISTA: ese identificador o lema NO existe (sin Mathlib hay muy pocos). No inventes nombres de lemas: usa tácticas "
                     "básicas (omega, simp, constructor, `exact ⟨testigo, by omega⟩`). «m es par» se escribe `m % 2 = 0` (o `∃ k, m = 2 * k`) y "
                     "«m es impar» `m % 2 = 1`. NO existen Even, Odd, Nat.Prime, Finset, Real ni la notación ℕ (usa Nat).")
PISTA_MATHLIB = ("PISTA: ese identificador o lema NO existe con ese nombre. No lo inventes: revisa el nombre, prueba `exact?`, o exprésalo con "
                 "operaciones básicas (`m % 2 = 0`) y tácticas como omega/simp.")
RE_DESCONOCIDO = re.compile(r"unknown (identifier|constant|namespace)|unknownidentifier|unknownconstant|function expected", re.I)
PISTA_SINTAXIS = "PISTA: error de sintaxis; revisa paréntesis, ⟨ ⟩, la sangría (todas las tácticas con la misma) y que cada táctica esté en su línea."

GUIA_LEAN = """REGLAS DE LEAN 4:
- Demuestra EXACTAMENTE el enunciado dado: no lo cambies, no añadas hipótesis, no inventes resultados.
- Prohibido: sorry, admit, axiom, native_decide, set_option, macros. Solo tácticas (:= by ...).
- Una táctica por línea, todas con la misma sangría.
TÁCTICAS ÚTILES:
- omega: aritmética lineal sobre Nat/Int (+, -, * por constantes, /, %, <, <=, =, y, o, no).
- simp, simp_all, decide, rfl, exact h, assumption, constructor (para y / si-y-solo-si).
- intro n / intros; obtain ⟨a, b⟩ := h; rcases h with ⟨a, b⟩ | c.
- Existenciales: `exact ⟨testigo, by omega⟩` (las tuplas planas SÍ valen: ∃ x, P x ∧ Q x se prueba con
  `exact ⟨t, by omega, by omega⟩` o con `refine ⟨t, ?_, ?_⟩`).
- Inducción: `induction n with | zero => ... | succ k ih => ...`.
- exact? busca un lema que cierre el objetivo (úsalo si no sabes cuál aplicar).
- SIN Mathlib no existen Even, Odd, Nat.Prime, Finset, Real: «par» es `m % 2 = 0`, «impar» `m % 2 = 1`."""


class LeanVerificador(Verificador):
    nombre = "lean"
    extension = "lean"
    admite_plan = True
    sistema = ("Eres un experto en matemáticas y en Lean 4. No inventes hipótesis ni resultados. "
               "No afirmes que algo funciona sin que el compilador lo haya comprobado.")

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.mathlib = bool(cfg.mathlib or cfg.lake_dir)
        self.lean_bin = cfg.lean or shutil.which("lean") or str(Path.home() / ".elan" / "bin" / "lean")

    def entorno(self) -> str:
        cmd = self._cmd(Path("x.lean"))[:-1] + ["--version"]
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=60, cwd=self.cfg.lake_dir or None)
        except (OSError, subprocess.TimeoutExpired) as e:
            raise RuntimeError(f"No se puede ejecutar Lean 4 ({self.lean_bin}): {e}")
        if r.returncode != 0:
            raise RuntimeError("Lean 4 no responde: " + (r.stdout + r.stderr)[:200])
        return (r.stdout or r.stderr).strip().splitlines()[0]

    def _cmd(self, archivo: Path) -> List[str]:
        if self.cfg.lake_dir:
            lake = shutil.which("lake") or str(Path.home() / ".elan" / "bin" / "lake")
            return [lake, "env", self.lean_bin, str(archivo)]
        return [self.lean_bin, str(archivo)]

    @property
    def encabezado(self) -> str:
        # autoImplicit desactivado: un identificador desconocido da "unknown identifier 'even'" (claro) en lugar de
        # convertirse en una variable implícita y producir "Function expected" (confuso para el modelo).
        return ("import Mathlib" if self.mathlib else "import Lean") + "\nset_option autoImplicit false"

    def tacticas_auto(self) -> List[str]:
        base = ["omega", "(simp; done)", "(simp_all; done)", "decide", "rfl", "(intros; omega)", "(intros; simp_all; done)",
                "(constructor <;> omega)", "(constructor <;> simp)"]
        if self.mathlib:
            base += ["(norm_num; done)", "linarith", "nlinarith [sq_nonneg _]", "positivity", "(ring; done)", "aesop"]
        return base + ["exact?"]

    def comprobar(self, codigo: str, nombre_final: str = "", permitir_sorry: bool = False) -> Tuple[bool, str]:
        if nombre_final:
            codigo = codigo.rstrip() + f"\n\n#print axioms {nombre_final}\n"
        with tempfile.TemporaryDirectory(prefix="demo_lean_") as td:
            f = Path(td) / "Problema.lean"
            f.write_text(codigo, encoding="utf-8")
            try:
                r = subprocess.run(self._cmd(f), capture_output=True, text=True, timeout=self.cfg.timeout_lean,
                                   cwd=self.cfg.lake_dir or td)
            except subprocess.TimeoutExpired:
                return False, f"Lean superó el límite de {self.cfg.timeout_lean}s."
            except OSError as e:
                return False, f"no se pudo ejecutar Lean: {e}"
        salida = ((r.stdout or "") + "\n" + (r.stderr or "")).strip()
        hay_error = r.returncode != 0 or re.search(r":\d+:\d+: error", salida) is not None
        if hay_error:
            return False, salida or "Lean falló sin mensaje."
        if not permitir_sorry and re.search(r"declaration uses [`'’]sorry[`'’]", salida):
            return False, "El código usa sorry (no es una demostración)."
        if nombre_final:
            m = re.search(r"depends on axioms: \[(.*?)\]", salida, flags=re.S)
            if m:
                usados = {a.strip() for a in m.group(1).split(",") if a.strip()}
                raros = usados - AXIOMAS_PERMITIDOS
                if raros:
                    return False, f"La demostración depende de axiomas no permitidos: {sorted(raros)}"
            elif "does not depend on any axioms" not in salida:
                return False, "No se pudo comprobar los axiomas de la demostración:\n" + recortar(salida, 400)
        return True, salida

    def prohibido(self, texto: str) -> Optional[str]:
        m = LEAN_PROHIBIDO.search(quitar_comentarios_lean(texto))
        return m.group(0).strip() if m else None

    # --- extracción
    def _codigo(self, texto: str) -> str:
        return elegir_bloque(texto, ("lean4", "lean"), lambda b: re.search(r"\b(theorem|lemma)\b|:=\s*by", b) is not None)

    def extraer_firma(self, texto: str) -> str:
        c = quitar_comentarios_lean(self._codigo(texto))
        m = re.search(r"\b(?:theorem|lemma)\s+\S", c)
        if not m:
            return ""
        c = c[m.start():]
        c = re.sub(r"^lemma\b", "theorem", c, count=1)
        i = c.find(":=")
        if i < 0:
            return ""
        firma = " ".join(c[:i].split())
        return (firma + " := by") if firma.startswith("theorem ") else ""

    def extraer_cuerpo(self, texto: str) -> str:
        c = self._codigo(texto)
        m = re.search(r"\b(?:theorem|lemma)\b.*?:=\s*by\b", c, flags=re.S)
        if m:
            resto = c[m.end():]
            lineas = resto.split("\n")
            primera = lineas[0].strip()
            cola: List[str] = []
            for l in lineas[1:]:
                if l.strip() and not l.startswith((" ", "\t")) and LEAN_ORDENES.match(l.strip()):   # otro comando (las tácticas sin sangría se aceptan)
                    break
                cola.append(l)
            cola = textwrap.dedent("\n".join(cola)).split("\n") if cola else []
            cuerpo = ([primera] if primera else []) + [l for l in cola]
        else:
            m2 = re.search(r":=\s*by\b", c)
            resto = c[m2.end():] if m2 else c
            cuerpo = textwrap.dedent(resto).split("\n")
        while cuerpo and not cuerpo[-1].strip():
            cuerpo.pop()
        while cuerpo and not cuerpo[0].strip():
            cuerpo.pop(0)
        cuerpo = [l.rstrip() for l in cuerpo]
        return sangrar(cuerpo) if cuerpo else ""

    def nombre(self, firma: str) -> str:
        m = re.match(r"\s*theorem\s+([^\s:({\[]+)", firma)
        return m.group(1) if m else ""

    def renombrar(self, firma: str, nuevo: str) -> str:
        return re.sub(r"^\s*theorem\s+[^\s:({\[]+", f"theorem {nuevo}", firma, count=1)

    def clave(self, firma: str) -> str:
        t = re.sub(r"^\s*(?:theorem|lemma)\s+\S+", "", normalizar(firma))
        return re.sub(r"\s+", "", re.sub(r":=\s*by$", "", t))

    def enunciado_trivial(self, firma: str) -> Optional[str]:
        t = normalizar(firma)
        t = re.sub(r"^theorem\s+\S+", "", t)
        t = re.sub(r":=\s*by$", "", t).strip()
        if re.search(r":\s*True\s*$", t) or re.search(r"\(\s*\w+\s*:\s*False\s*\)", t) or t.endswith("= True"):
            return "el enunciado es trivial (True) o tiene una hipótesis imposible (False)"
        return None

    def bloque(self, firma: str, cuerpo: str) -> str:
        return f"{firma}\n{cuerpo}"

    def estrategia(self, firma: str) -> str:
        """Consejos según la FORMA del objetivo (un modelo pequeño los necesita)."""
        t = firma.split(":=")[0]
        h = []
        if "∃" in t:
            h.append("El objetivo es un EXISTENCIAL: elige un testigo CONCRETO en función de las variables (p. ej. `2 * n + 2`) y escribe "
                     "`exact ⟨testigo, by omega⟩`. NO busques lemas de la biblioteca ni uses `obtain` sobre lemas que no conozcas.")
        if "↔" in t:
            h.append("Es un si-y-solo-si: `constructor`, y en cada lado `intro h` y después `omega` (o `simp_all`).")
        if "→" in t or "->" in t:
            h.append("Hay una implicación: empieza con `intro h` y usa `h` (con `omega` o `exact`).")
        if "%" in t or "/" in t:
            h.append("`omega` entiende `%` y `/` por constantes: úsalo para cerrar las igualdades y desigualdades.")
        return "\n".join("- " + x for x in h)

    def testigos_auto(self, firma: str) -> List[str]:
        """Alternativas `exact ⟨W, by omega⟩` para objetivos existenciales: prueba testigos lineales típicos sin gastar el modelo."""
        t = firma.split(":=")[0]
        if "∃" not in t:
            return []
        vars_: List[str] = []
        for m in re.finditer(r"\(([^():]+?)\s*:\s*(?:Nat|Int)\)", t):
            vars_ += m.group(1).split()
        cands: List[str] = []
        for v in vars_[:2]:
            cands += [v, f"{v} + 1", f"{v} + 2", f"2 * {v}", f"2 * {v} + 1", f"2 * {v} + 2", f"3 * {v}", f"{v} * {v}", f"{v} * {v} + 1"]
        cands += ["0", "1", "2"]
        vistos, alts = set(), []
        for w in cands:
            if w not in vistos:
                vistos.add(w)
                alts.append(f"exact ⟨{w}, by omega⟩")
        return alts

    def codigo(self, estado: Dict[str, Any], bloques: List[str], firma: str = "", cuerpo: str = "") -> str:
        return ensamblar(self.encabezado, estado.get("definiciones", ""), *bloques, f"{firma}\n{cuerpo}" if firma else "")

    def auto(self, estado: Dict[str, Any], firma: str, bloques: List[str]) -> Optional[str]:
        alts = self.tacticas_auto()
        alts = alts[:-1] + self.testigos_auto(firma) + alts[-1:]     # `exact?` (lenta) siempre la última
        cuerpo = "  first\n" + "\n".join(f"  | {a}" for a in alts)
        ok, _ = self.comprobar(self.codigo(estado, bloques, firma, cuerpo))
        return cuerpo if ok else None

    def resumir_error(self, salida: str) -> str:
        if not salida:
            return "error desconocido"
        trozos = re.split(r"(?m)^(?=\S+:\d+:\d+: (?:error|warning))", salida)
        errores = [t.strip() for t in trozos if re.match(r"\S+:\d+:\d+: error", t.strip())]
        base = "\n".join(errores[:3]) if errores else salida
        base = re.sub(r"\S*Problema\.lean:", "línea ", base)
        base = recortar(base, 1400)
        if RE_DESCONOCIDO.search(salida):
            base += "\n" + (PISTA_MATHLIB if self.mathlib else PISTA_SIN_MATHLIB)
        elif re.search(r"unexpected token|expected term|expected '", salida):
            base += "\n" + PISTA_SINTAXIS
        return base

    def titulo_error(self, resumen: str) -> str:
        ls = [l.strip() for l in (resumen or "").splitlines() if l.strip() and not l.strip().startswith("PISTA")]
        return " ".join(ls[:3])

    # --- prompts
    def prompt_formalizar(self, problema: str, error: str) -> str:
        libs = ("Puedes usar Mathlib (Real, Finset, Even, Nat.Prime...)." if self.mathlib else
                "Solo hay Lean 4 SIN Mathlib: usa Nat e Int (no Real) y operaciones básicas. «m es par» se escribe `m % 2 = 0` "
                "(NO existen Even, Odd, Nat.Prime, Finset ni la notación ℕ: usa Nat).")
        return f"""Traduce el siguiente problema matemático a un ENUNCIADO de Lean 4 (sin demostrarlo).

PROBLEMA:
{problema}

REGLAS:
- Conserva exactamente el significado matemático. No lo conviertas en True ni añadas hipótesis.
- {libs}
- Incluye solo definiciones realmente necesarias. No uses axiomas ni sorry.
- El teorema debe terminar en `:= by`.

ERROR DEL INTENTO ANTERIOR (si lo hay):
{error or '(ninguno)'}

Devuelve SOLO este JSON:
{{"definiciones": "", "teorema": "theorem nombre (...) : proposicion := by"}}"""

    def prompt_fidelidad(self, problema: str, defs: str, firma: str) -> str:
        return f"""Comprueba si este enunciado de Lean 4 expresa EXACTAMENTE el problema (ni más fuerte, ni más débil, ni trivial).

PROBLEMA:
{problema}

DEFINICIONES:
{defs or '(ninguna)'}

ENUNCIADO LEAN:
{firma}

Devuelve SOLO JSON: {{"equivalente": true|false, "motivo": "una frase"}}"""

    ejemplos = ""

    def prompt_prueba(self, firma: str, bloques: List[str], estado: Dict[str, Any], nodo: Dict[str, Any], tipo: str) -> str:
        ctx = "\n\n".join(bloques) or "(ninguno)"
        prev = ""
        if nodo.get("ultimo_codigo"):
            prev = f"""
TU INTENTO ANTERIOR (falló):
```lean
{nodo['ultimo_codigo']}
```
ERROR DE LEAN:
{nodo.get('ultimo_error', '')}
Corrige ese intento: analiza el error y cambia lo necesario (no repitas lo mismo)."""
        otros = ""
        if len(nodo.get("errores", [])) > 1:
            otros = "\nERRORES DE INTENTOS ANTERIORES (para no repetirlos):\n" + "\n---\n".join(nodo["errores"][-4:-1])
        antes = nodo.get("cuerpos", [])[:-1][-2:]
        if antes:
            otros += "\nDEMOSTRACIONES QUE YA FALLARON (NO las repitas ni las variantes triviales):\n" + "\n---\n".join(
                "```lean\n" + c + "\n```" for c in antes)
        est = self.estrategia(firma)
        if est:
            otros += "\n\nESTRATEGIA SUGERIDA PARA ESTE OBJETIVO:\n" + est
        return f"""Demuestra este teorema en Lean 4.

TEOREMA:
```lean
{firma}
```
DEFINICIONES:
{estado.get('definiciones') or '(ninguna)'}

LEMAS YA DEMOSTRADOS (puedes usarlos por su nombre):
{ctx}
{self.ejemplos}{prev}{otros}

{GUIA_LEAN}

Devuelve el teorema completo en UN solo bloque ```lean ... ``` (con `:= by` y la demostración)."""

    def prompt_plan(self, firma: str, bloques: List[str], estado: Dict[str, Any], nodo: Dict[str, Any], max_lemas: int,
                    fallidos: List[str]) -> str:
        probados = "\n\n".join(bloques) or "(ninguno)"
        f = "\n".join(f"- {x}" for x in fallidos[-6:]) or "(ninguno)"
        return f"""Diseña una estrategia para demostrar este teorema en Lean 4 descomponiéndolo en lemas.

TEOREMA PRINCIPAL:
{firma}

DEFINICIONES:
{estado.get('definiciones') or '(ninguna)'}

LEMAS YA DEMOSTRADOS (no los repitas; úsalos):
{probados}

LEMAS QUE NO SE PUDIERON DEMOSTRAR (propón otros, más sencillos o distintos):
{f}

ÚLTIMO ERROR DEL TEOREMA PRINCIPAL:
{nodo.get('ultimo_error') or '(ninguno)'}

Propón como máximo {max_lemas} lemas NUEVOS, en el orden en que deben demostrarse.
REGLAS: cada lema aporta una propiedad intermedia necesaria; no repitas el teorema principal; sin lemas tautológicos;
sin hipótesis injustificadas; sin sorry/admit/axiom; cada uno termina en `:= by`.

Devuelve SOLO JSON: {{"lemas": ["theorem nombre (...) : proposicion := by"]}}"""


# ------------------------------------------------------------------ biblioteca de demostraciones verificadas
def _tokens(t: str) -> set:
    return {w for w in re.findall(r"[a-zA-Z_áéíóúñ]{3,}|\d+", (t or "").lower())}


def biblioteca_cargar(ruta: str) -> List[Dict[str, Any]]:
    try:
        d = json.loads(Path(ruta).read_text(encoding="utf-8")) if ruta and Path(ruta).is_file() else []
        return d if isinstance(d, list) else []
    except (OSError, ValueError):
        return []


def biblioteca_anadir(ruta: str, entrada: Dict[str, Any]) -> None:
    if not ruta:
        return
    datos = [e for e in biblioteca_cargar(ruta) if e.get("id") != entrada.get("id")]
    datos.append(entrada)
    Path(ruta).parent.mkdir(parents=True, exist_ok=True)
    escribir_atomico(Path(ruta), json.dumps(datos, ensure_ascii=False, indent=1))


def biblioteca_ejemplos(ruta: str, enunciado: str, backend: str, n: int = 2, maximo: int = 900) -> str:
    """Las n demostraciones verificadas más parecidas (solapamiento de palabras): ejemplos reales que el modelo imita."""
    q = _tokens(enunciado)
    pun = []
    for e in biblioteca_cargar(ruta):
        if e.get("backend") != backend or not e.get("codigo") or e.get("enunciado") == enunciado:
            continue
        t = _tokens(e.get("enunciado", "")) | set(e.get("etiquetas", []))
        pun.append((len(q & t) / (len(q | t) or 1), e))
    pun.sort(key=lambda x: -x[0])
    out = []
    for sc, e in pun[:n]:
        if sc <= 0:
            continue
        out.append(f"PROBLEMA RESUELTO: {recortar(e['enunciado'], 220)}\n```{'lean' if backend == 'lean' else 'python'}\n{recortar(e['codigo'], maximo)}\n```")
    return ("\nEJEMPLOS VERIFICADOS (de problemas parecidos ya resueltos; imita su estilo, no los copies):\n" + "\n\n".join(out) + "\n") if out else ""


# ------------------------------------------------------------------ SymPy (física y cálculo simbólico)
PRELUDIO_PY = '''import sympy as sp
from sympy import *

def verificar(i, cond):
    if cond is True or cond == sp.true:
        pass
    elif cond is False or cond == sp.false:
        raise AssertionError("CRITERIO_FALLA %s: el criterio vale False. Ojo: '==' compara la FORMA, no el valor; compara con simplify(a - b) == 0. Mira los VALORES impresos arriba." % i)
    else:
        raise AssertionError("CRITERIO_FALLA %s: el criterio no es booleano (vale %s). Usa una comparacion, p. ej. simplify(a - b) == 0." % (i, cond))
    print("CRITERIO_OK %s" % i)
'''
PY_PROHIBIDO = re.compile(
    r"\b(verificar|subprocess|socket|urllib|requests|http|ftplib|ctypes|shutil|pathlib|importlib|__import__|exec|eval|"
    r"compile|globals|locals|breakpoint|input|open|os\.system|os\.popen|os\.remove|sys\.exit|exit|quit|"
    r"os|sys|SystemExit|_exit|builtins|getattr|setattr|sympify|parse_expr|__\w+__)\b")


class SympyVerificador(Verificador):
    nombre = "sympy"
    extension = "py"
    admite_plan = False
    sistema = ("Eres un físico teórico y matemático riguroso que programa en Python con SymPy. DERIVAS los resultados "
               "paso a paso a partir de principios; nunca los das por supuestos ni los escribes directamente.")

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg

    def entorno(self) -> str:
        r = subprocess.run([sys.executable, "-c", "import sympy; print(sympy.__version__)"], capture_output=True, text=True, timeout=60)
        if r.returncode != 0:
            raise RuntimeError("Falta SymPy: pip install sympy")
        return "sympy " + r.stdout.strip()

    def comprobar(self, codigo: str, nombre_final: str = "") -> Tuple[bool, str]:
        limpio = {k: v for k, v in os.environ.items() if not re.search(r"(KEY|TOKEN|SECRET|PASS|CRED|AUTH)", k, re.I)}
        kw: Dict[str, Any] = {}
        if os.name == "posix":
            def limites() -> None:   # noqa: ANN202
                import resource
                resource.setrlimit(resource.RLIMIT_CPU, (self.cfg.timeout_py + 5, self.cfg.timeout_py + 10))
                resource.setrlimit(resource.RLIMIT_AS, (6 * 2**30, 6 * 2**30))
            kw["preexec_fn"] = limites
        with tempfile.TemporaryDirectory(prefix="demo_py_") as td:
            f = Path(td) / "derivacion.py"
            f.write_text(codigo, encoding="utf-8")
            try:
                r = subprocess.run([sys.executable, "-B", str(f)], capture_output=True, text=True, timeout=self.cfg.timeout_py,
                                   cwd=td, env=limpio, **kw)
            except subprocess.TimeoutExpired:
                return False, f"El cálculo superó el límite de {self.cfg.timeout_py}s."
        salida = ((r.stdout or "") + "\n" + (r.stderr or "")).strip()
        if r.returncode < 0:
            return False, (f"El proceso fue terminado por el sistema (señal {-r.returncode}): probablemente superó el límite de "
                           f"CPU o de memoria. Usa un cálculo más ligero (menos simplificaciones pesadas, series cortas).")
        if r.returncode != 0:
            return False, salida or "El script falló sin mensaje."
        if nombre_final and not (salida.count("TODO_VERIFICADO") == 1 and salida.splitlines()[-1].strip() == "TODO_VERIFICADO"):
            return False, "El script terminó pero no llegó a comprobar todos los criterios."
        return True, salida

    def prohibido(self, texto: str) -> Optional[str]:
        m = PY_PROHIBIDO.search(quitar_comentarios_py(texto))
        return m.group(0) if m else None

    def _codigo(self, texto: str) -> str:
        return elegir_bloque(texto, ("python", "py"), lambda b: "import" in b or "=" in b)

    def extraer_cuerpo(self, texto: str) -> str:
        c = self._codigo(texto)
        c = "\n".join(l for l in c.split("\n") if not re.match(r"\s*(import sympy|from sympy import \*)\s*$", l))
        return textwrap.dedent(c).strip("\n")

    def bloque(self, firma: str, cuerpo: str) -> str:
        return cuerpo

    def codigo(self, estado: Dict[str, Any], bloques: List[str], firma: str = "", cuerpo: str = "") -> str:
        crit = "\n".join(f"verificar({i + 1}, {c})" for i, c in enumerate(estado.get("criterios", [])))
        nombres = [v for v in estado.get("variables", []) if re.fullmatch(r"\w+", str(v))]
        vals = ("for _n in %r:\n    print('VALOR', _n, '=', globals().get(_n, 'SIN DEFINIR'))\n" % (nombres,)) if nombres else ""
        return ensamblar(PRELUDIO_PY, estado.get("preambulo", ""), *bloques, cuerpo, "# --- criterios fijados antes de resolver ---\n" + vals + crit,
                         'print("TODO_VERIFICADO")')

    def auto(self, estado: Dict[str, Any], firma: str, bloques: List[str]) -> Optional[str]:
        return None

    def resumir_error(self, salida: str) -> str:
        lineas = [l for l in (salida or "").splitlines() if l.strip()]
        return recortar("\n".join(lineas[-12:]) or "error desconocido", 1400)

    def titulo_error(self, resumen: str) -> str:
        lineas = [l for l in (resumen or "").splitlines() if l.strip()]
        return lineas[-1] if lineas else ""     # la excepción es la última línea del traceback

    def ayuda_error(self, salida: str) -> str:
        """Consejo accionable según la excepción (el modelo pequeño no suele interpretar tracebacks)."""
        t = salida or ""
        if re.search(r"NameError", t):
            return ("\nPISTA: usas un nombre sin definir. Declara TODO símbolo con symbols(...) en el preámbulo; "
                    "el preámbulo NO puede usar las variables de resultado (esas las define la solución).")
        if re.search(r"SyntaxError", t):
            return "\nPISTA: error de sintaxis Python; una instrucción por línea, sin texto explicativo ni marcas de formato."
        if re.search(r"TypeError|AttributeError", t):
            return "\nPISTA: usa funciones de SymPy (sqrt, Rational, Eq, symbols) y no métodos inventados."
        if re.search(r"ImportError|ModuleNotFoundError", t):
            return "\nPISTA: solo se dispone de sympy (ya importado con `from sympy import *`); no importes nada más."
        return ""

    def enunciado_trivial(self, firma: str) -> Optional[str]:
        return None

    def prompt_formalizar(self, problema: str, error: str) -> str:
        return f"""Convierte este problema de física/matemáticas en una ESPECIFICACIÓN verificable con SymPy.

PROBLEMA:
{problema}

EJEMPLO COMPLETO (para otro problema: caída libre) de la respuesta esperada:
{{"descripcion": "Deducir la velocidad final v de un cuerpo que cae una altura h con gravedad g.",
 "preambulo": "g, h, m = symbols('g h m', positive=True)",
 "variables": ["v"],
 "criterios": ["simplify(v**2 - 2*g*h) == 0"]}}
(`preambulo` solo declara símbolos con `symbols(...)`; NO calcula nada ni usa las variables de resultado; `variables` son los nombres que luego definirá la solución;
 cada criterio es UNA expresión Python que vale True; no uses f-strings, `print`, `import` ni texto fuera del JSON.)

Debes definir:
1. "preambulo": código Python (SymPy) que declara símbolos, constantes y supuestos (p. ej. `G, M, c, r = symbols('G M c r', positive=True)`).
2. "variables": nombres de las variables que la solución debe DEFINIR (los resultados).
3. "criterios": expresiones Python que valen True solo si el resultado es correcto (p. ej. `simplify(v_esc.subs(r, 2*G*M/c**2) - c) == 0`).
   Deben usar las variables de resultado, ser estrictas (no triviales) y comprobar el resultado pedido, no uno parecido.
4. "descripcion": el objetivo en una frase.

ERROR DEL INTENTO ANTERIOR (si lo hay):
{error or '(ninguno)'}

Devuelve SOLO JSON: {{"descripcion": "", "preambulo": "", "variables": ["..."], "criterios": ["..."]}}"""

    def prompt_fidelidad(self, problema: str, defs: str, firma: str) -> str:
        return f"""Comprueba si estos criterios verifican EXACTAMENTE lo que pide el problema (ni más débiles, ni triviales, ni de otro resultado).

PROBLEMA:
{problema}

{defs}

Devuelve SOLO JSON: {{"equivalente": true|false, "motivo": "una frase"}}"""

    ejemplos = ""

    def prompt_prueba(self, firma: str, bloques: List[str], estado: Dict[str, Any], nodo: Dict[str, Any], tipo: str) -> str:
        prev = ""
        if nodo.get("ultimo_codigo"):
            prev = f"""
TU INTENTO ANTERIOR (falló):
```python
{nodo['ultimo_codigo']}
```
SALIDA / ERROR:
{nodo.get('ultimo_error', '')}
Corrige el fallo concreto; no repitas lo mismo."""
        crit = "\n".join(f"  {i + 1}. {c}" for i, c in enumerate(estado.get("criterios", [])))
        return f"""Resuelve este problema DERIVANDO el resultado con SymPy.

PROBLEMA:
{estado['problema']['enunciado']}

OBJETIVO: {firma}

CONTEXTO ya ejecutado antes de tu código (no lo repitas):
```python
{estado.get('preambulo', '')}
```
Tu código debe DEFINIR estas variables: {', '.join(estado.get('variables', []))}
Después de tu código se comprobarán automáticamente estos criterios (no los escribas tú):
{crit}

{self.ejemplos}EJEMPLO de solución (otro problema: velocidad final v de una caída libre desde la altura h; símbolos g, h, m, v ya declarados):
```python
E = Rational(1, 2)*m*v**2 - m*g*h
v_fall = solve(Eq(E, 0), v)[0]
```
REGLAS: deriva el resultado (planteando ecuaciones y resolviendo con solve/dsolve/integrate/series/simplify...), no lo escribas ya hecho;
`solve` devuelve una LISTA: toma el elemento con [0] (los símbolos con positive=True dan una sola raíz); no redeclares los símbolos del contexto;
sin entrada/salida de ficheros ni red; sin `verificar`; el código debe ejecutarse sin errores.
{prev}

Devuelve SOLO un bloque ```python ... ```."""

    def prompt_plan(self, *a: Any, **k: Any) -> str:
        return ""


# =============================================================================
# 5. MOTOR: árbol de demostración persistente y reanudable
# =============================================================================
TEMPERATURAS = [0.1, 0.25, 0.4, 0.55, 0.7, 0.85]


class Motor:
    def __init__(self, cfg: Config, llm: Any, ver: Verificador, directorio: Path, log: Optional[Callable[[str], None]] = None) -> None:
        self.cfg, self.llm, self.ver = cfg, llm, ver
        self.dir = Path(directorio)
        self.dir.mkdir(parents=True, exist_ok=True)
        self.log = log or (lambda s: print(s, flush=True))
        self.estado: Dict[str, Any] = {}
        self.t_inicio = time.time()
        self.t_base = 0.0
        self.llamadas_sesion = 0
        self.parar = False

    # ------------------------------------------------------------ persistencia
    @property
    def ruta_estado(self) -> Path:
        return self.dir / "estado.json"

    def nuevo_estado(self, problema: Dict[str, Any]) -> Dict[str, Any]:
        return {"version": ESTADO_VERSION, "problema": problema, "huella": self.huella_problema(problema), "creado": ahora(),
                "estado": "formalizando", "formalizado": False, "definiciones": "", "preambulo": "", "variables": [], "criterios": [],
                "raiz": None, "contador": 0, "llamadas": 0, "tiempo": 0.0, "intentos_formalizar": 0, "ultimo_error": "",
                "resultado": {"codigo": "", "archivo": ""}}

    def huella_problema(self, problema: Dict[str, Any]) -> str:
        return huella(problema["enunciado"] + "|" + problema.get("backend", "lean") + "|" + str(bool(self.cfg.mathlib or self.cfg.lake_dir)))

    def cargar(self, problema: Dict[str, Any], reiniciar: bool = False) -> None:
        if reiniciar and self.ruta_estado.exists():
            self.ruta_estado.rename(self.dir / f"estado.{int(time.time())}.bak.json")
        if self.ruta_estado.exists():
            try:
                datos = json.loads(self.ruta_estado.read_text(encoding="utf-8"))
                if datos.get("version") == ESTADO_VERSION and datos.get("huella") == self.huella_problema(problema):
                    self.estado = datos
                    self.t_base = float(datos.get("tiempo", 0.0))
                    self.log(f"Reanudando: {datos['estado']}, {datos['llamadas']} llamadas y {self.t_base / 60:.1f} min de trabajo previos.")
                    if datos["estado"] == "fallido":
                        self.reabrir_fallidos()
                    return
                self.log("El estado guardado es de otro problema o versión: se conserva como copia y se empieza de nuevo.")
                self.ruta_estado.rename(self.dir / f"estado.{int(time.time())}.bak.json")
            except (OSError, json.JSONDecodeError) as e:
                self.log(f"Estado ilegible ({e}); se empieza de nuevo.")
                self.ruta_estado.rename(self.dir / f"estado.{int(time.time())}.corrupto.json")
        self.estado = self.nuevo_estado(problema)
        self.guardar()

    def reabrir_fallidos(self) -> None:
        """Un estado "fallido" significa que se agotaron los límites. Al volver a ejecutar se reintenta: los contadores
        vuelven a cero, pero se conservan los lemas demostrados, el historial de errores y las demostraciones ya
        descartadas (huellas), para no repetir lo que ya falló."""
        e = self.estado
        e["estado"] = "resolviendo" if e.get("formalizado") else "formalizando"
        e["intentos_formalizar"] = 0
        e.pop("error_fatal", None)

        def abrir(n: Dict[str, Any]) -> None:
            if n["estado"] == "fallido":
                n.update(estado="pendiente", directos=0, finales=0, replanes=0, necesita_plan=True)
            for h in n["hijos"]:
                abrir(h)
        if e.get("raiz"):
            abrir(e["raiz"])
        self.log("La ejecución anterior terminó sin resolverse: se reintenta con los contadores a cero "
                 "(se conservan los lemas demostrados y las demostraciones ya descartadas).")
        self.guardar()

    def guardar(self) -> None:
        self.estado["tiempo"] = self.t_base + (time.time() - self.t_inicio)
        escribir_atomico(self.ruta_estado, json.dumps(self.estado, ensure_ascii=False, indent=1))

    def evento(self, nombre_evento: str, /, **datos: Any) -> None:
        rec = {"ts": ahora(), "t": round(self.estado.get("tiempo", 0.0), 1), "evento": nombre_evento, **datos}
        with open(self.dir / "registro.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    def say(self, n: Optional[Dict[str, Any]], msg: str) -> None:
        t = self.t_base + (time.time() - self.t_inicio)
        h, m, s = int(t // 3600), int(t % 3600 // 60), int(t % 60)
        self.log(f"[{h:02d}:{m:02d}:{s:02d} | llamadas {self.estado.get('llamadas', 0)}{' | ' + n['id'] if n else ''}] {msg}")

    # ------------------------------------------------------------ presupuesto y modelo
    def presupuesto(self) -> None:
        if self.parar:
            raise Pausa("parada solicitada")
        if self.cfg.horas and (time.time() - self.t_inicio) >= self.cfg.horas * 3600:
            raise Pausa(f"límite de {self.cfg.horas} h de esta sesión")
        if self.cfg.max_intentos and self.llamadas_sesion >= self.cfg.max_intentos:
            raise Pausa(f"límite de {self.cfg.max_intentos} llamadas al modelo en esta sesión")

    def _llm(self, prompt: str, temperatura: float, seed: int, json_mode: bool = False, num_predict: Optional[int] = None) -> str:
        self.presupuesto()
        self.estado["llamadas"] += 1
        self.llamadas_sesion += 1
        try:
            return self.llm.chat(self.ver.sistema, prompt, temperatura=temperatura, num_predict=num_predict or self.cfg.num_predict,
                                 seed=seed, json_mode=json_mode)
        finally:
            self.guardar()

    # ------------------------------------------------------------ FASE 0: formalización
    def formalizar(self) -> None:
        if self.estado["formalizado"]:
            return
        self.say(None, "FASE 0: formalización del enunciado")
        problema = self.estado["problema"]["enunciado"]
        error = self.estado.get("ultimo_error", "")
        espec = self.estado["problema"].get("especificacion")
        if isinstance(espec, dict) and self.cfg.backend == "sympy":      # especificación revisada por una persona: sin llamar al modelo
            try:
                firma, _ = self._validar_formalizacion(dict(espec))
                self.estado.update({"formalizado": True, "estado": "resolviendo", "raiz": self.nodo(firma, 0), "ultimo_error": ""})
                self.guardar()
                self.evento("formalizado", firma=firma, especificacion="incluida")
                self.say(None, "Especificación incluida (revisada, sin usar el modelo):\n" + firma + "\nCriterios: " + " ; ".join(self.estado["criterios"]))
                return
            except (ValueError, KeyError, TypeError) as e:
                self.say(None, "La especificación incluida no es válida (" + str(e).splitlines()[-1] + "); se pide al modelo.")
        while self.estado["intentos_formalizar"] < self.cfg.formalizar:
            self.estado["intentos_formalizar"] += 1
            i = self.estado["intentos_formalizar"]
            try:
                resp = self._llm(self.ver.prompt_formalizar(problema, error), 0.1 + 0.03 * i, self.cfg.semilla * 7919 + i, json_mode=True)
                datos = extraer_json(resp)
                firma, descripcion = self._validar_formalizacion(datos)
                if self.cfg.fidelidad:
                    self._fidelidad(problema, firma)
                self.estado["formalizado"] = True
                self.estado["estado"] = "resolviendo"
                self.estado["raiz"] = self.nodo(firma, 0)
                self.estado["ultimo_error"] = ""
                self.guardar()
                self.evento("formalizado", firma=firma)
                self.say(None, "Formalización aceptada:\n" + firma)
                return
            except (ValueError, KeyError, TypeError) as e:
                error = str(e)
                self.estado["ultimo_error"] = recortar(error, 800)
                self.guardar()
                self.evento("formalizacion_fallo", error=recortar(error, 400))
                lineas_e = [l for l in error.splitlines() if l.strip()]
                self.say(None, f"formalización {i}/{self.cfg.formalizar} rechazada: {recortar(lineas_e[-1] if lineas_e else '', 200)}")
        raise RuntimeError("No se pudo formalizar el problema: " + recortar(error, 300))

    def _validar_formalizacion(self, datos: Dict[str, Any]) -> Tuple[str, str]:
        ver = self.ver
        if isinstance(ver, LeanVerificador):
            defs = datos.get("definiciones", "") or ""
            defs = "\n".join(l for l in str(defs).splitlines() if not l.strip().startswith("import "))
            firma = ver.extraer_firma(str(datos.get("teorema", "")))
            if not firma:
                raise ValueError("No se obtuvo una firma `theorem ... := by` válida.")
            malo = ver.prohibido(defs + "\n" + firma)
            if malo:
                raise ValueError(f"Construcción prohibida en la formalización: {malo}")
            triv = ver.enunciado_trivial(firma)
            if triv:
                raise ValueError(triv)
            self.estado["definiciones"] = defs.strip()
            ok, salida = ver.comprobar(ver.codigo(self.estado, [], firma, "  sorry"), permitir_sorry=True)
            if not ok:
                raise ValueError(ver.resumir_error(salida))
            return firma, firma
        # SymPy
        pre = str(datos.get("preambulo", "") or "")
        crit = [str(c) for c in (datos.get("criterios") or []) if str(c).strip()]
        vars_ = [str(v) for v in (datos.get("variables") or [])]
        desc = str(datos.get("descripcion", "") or "").strip()
        if not crit or not vars_ or not desc:
            raise ValueError("Faltan 'descripcion', 'variables' o 'criterios'.")
        malo = ver.prohibido(pre) or next((m for m in (ver.prohibido(c) for c in crit) if m), None)
        if malo:
            raise ValueError(f"Construcción prohibida en la especificación: {malo}")
        try:
            compile(pre, "<preambulo>", "exec")
            for c in crit:
                compile(c, "<criterio>", "eval")
        except SyntaxError as e:
            raise ValueError(f"Error de sintaxis en la especificación: {e}")
        ok, salida = ver.comprobar(ensamblar(PRELUDIO_PY, pre, 'print("TODO_VERIFICADO")'), nombre_final="x")
        if not ok:
            raise ValueError("El preámbulo no se ejecuta.\nTU PREÁMBULO:\n" + recortar(pre, 500) + ver.ayuda_error(salida) +
                             "\nERROR:\n" + ver.resumir_error(salida))
        self.estado.update({"preambulo": pre.strip(), "variables": vars_, "criterios": crit})
        return desc, desc

    def _fidelidad(self, problema: str, firma: str) -> None:
        """Segunda opinión del modelo: ¿el enunciado formal dice lo que pide el problema? (señal débil, pero útil)."""
        ver = self.ver
        if isinstance(ver, LeanVerificador):
            prompt = ver.prompt_fidelidad(problema, self.estado.get("definiciones", ""), firma)
        else:
            prompt = ver.prompt_fidelidad(problema, "PREÁMBULO:\n" + self.estado.get("preambulo", "") + "\nCRITERIOS:\n" +
                                          "\n".join(self.estado.get("criterios", [])), firma)
        try:
            obj = extraer_json(self._llm(prompt, 0.0, self.cfg.semilla, json_mode=True, num_predict=300))
        except (ValueError, TypeError):
            return
        if obj.get("equivalente") is False:
            raise ValueError("Fidelidad: el enunciado formal no coincide con el problema (" + str(obj.get("motivo", ""))[:200] + ")")

    # ------------------------------------------------------------ árbol
    def nodo(self, firma: str, prof: int) -> Dict[str, Any]:
        self.estado["contador"] += 1
        return {"id": f"n{self.estado['contador']}", "firma": firma, "prof": prof, "estado": "pendiente", "cuerpo": "", "auto": False,
                "directos": 0, "finales": 0, "replanes": 0, "necesita_plan": True, "hijos": [], "fallidos": [], "errores": [],
                "ultimo_codigo": "", "ultimo_error": "", "huellas": [], "cuerpos": [], "repetidas": 0}

    def bloque_nodo(self, n: Dict[str, Any]) -> str:
        hijos = [self.bloque_nodo(h) for h in n["hijos"] if h["estado"] == "probado"]
        return ensamblar(*hijos, self.ver.bloque(n["firma"], n["cuerpo"]))

    def _fallo(self, n: Dict[str, Any], cuerpo: str, salida: str, tipo: str) -> bool:
        resumen = self.ver.resumir_error(salida)
        if "repetida" in resumen and n.get("ultimo_error"):      # conservar el error REAL para que el modelo pueda corregirlo
            previo = re.sub(r"^\(Repetiste[^\n]*\n", "", n["ultimo_error"])
            n["ultimo_codigo"], n["ultimo_error"] = cuerpo, "(Repetiste una respuesta idéntica a un intento anterior: CAMBIA de enfoque.)\n" + previo
        else:
            n["ultimo_codigo"], n["ultimo_error"] = cuerpo, resumen
        n["errores"] = (n["errores"] + [resumen])[-6:]
        if cuerpo and "repetida" not in resumen:
            n["cuerpos"] = (n.get("cuerpos", []) + [recortar(cuerpo, 500)])[-4:]
        self.guardar()
        self.evento("intento_fallido", nodo=n["id"], tipo=tipo, error=recortar(resumen, 400))
        self.say(n, f"{tipo} {n['directos'] if tipo == 'directo' else n['finales']}: FALLA — {recortar(self.ver.titulo_error(resumen), 130)}")
        return False

    def _aceptar(self, n: Dict[str, Any], cuerpo: str, tipo: str) -> bool:
        n["cuerpo"], n["estado"] = cuerpo, "probado"
        n["ultimo_error"] = ""
        self.guardar()
        self.evento("probado", nodo=n["id"], tipo=tipo, firma=recortar(n["firma"], 200))
        self.say(n, f"{tipo}: VERIFICADO — {recortar(n['firma'].splitlines()[0], 110)}")
        return True

    def _intento(self, n: Dict[str, Any], ctx: List[str], tipo: str) -> bool:
        self.presupuesto()
        clave = "directos" if tipo == "directo" else "finales"
        n[clave] += 1
        idx = n["directos"] + n["finales"] + 6 * n["replanes"]
        temp = min(1.0, TEMPERATURAS[min(idx - 1, len(TEMPERATURAS) - 1)] + 0.15 * n.get("repetidas", 0))   # si repite, más variedad
        seed = self.cfg.semilla * 1000003 + int(n["id"][1:]) * 1009 + n["directos"] * 31 + n["finales"] * 17 + n["replanes"] * 5
        resp = self._llm(self.ver.prompt_prueba(n["firma"], ctx, self.estado, n, tipo), temp, seed)
        cuerpo = self.ver.extraer_cuerpo(resp)
        if not cuerpo:
            return self._fallo(n, "", "No se pudo extraer una demostración de la respuesta.", tipo)
        malo = self.ver.prohibido(cuerpo)
        if malo:
            return self._fallo(n, cuerpo, f"Construcción prohibida: {malo}", tipo)
        h = huella(cuerpo)
        if h in n["huellas"]:
            n["repetidas"] = n.get("repetidas", 0) + 1
            return self._fallo(n, cuerpo, "Demostración repetida: ya se probó y falló.", tipo)
        n["huellas"] = (n["huellas"] + [h])[-200:]
        ok, salida = self.ver.comprobar(self.ver.codigo(self.estado, ctx, n["firma"], cuerpo), nombre_final="x" if self.ver.nombre == "sympy" else "")
        return self._aceptar(n, cuerpo, tipo) if ok else self._fallo(n, cuerpo, salida, tipo)

    def _planificar(self, n: Dict[str, Any], ctx: List[str]) -> List[Dict[str, Any]]:
        ver = self.ver
        assert isinstance(ver, LeanVerificador)
        existentes = {ver.clave(h["firma"]) for h in n["hijos"]} | {ver.clave(f) for f in n["fallidos"]} | {ver.clave(n["firma"])}
        probados = ctx + [self.bloque_nodo(h) for h in n["hijos"] if h["estado"] == "probado"]
        error = ""
        for k in range(1, self.cfg.planes + 1):
            seed = self.cfg.semilla * 2000003 + int(n["id"][1:]) * 1013 + n["replanes"] * 101 + k
            try:
                obj = extraer_json(self._llm(ver.prompt_plan(n["firma"], probados, self.estado, n, self.cfg.max_lemas, n["fallidos"]),
                                             0.25 + 0.05 * (k + n["replanes"]), seed, json_mode=True, num_predict=2000))
                lista = obj.get("lemas")
                if not isinstance(lista, list):
                    raise ValueError("'lemas' debe ser una lista")
                firmas, claves = [], set(existentes)
                for raw in lista[: self.cfg.max_lemas]:
                    f = ver.extraer_firma(str(raw))
                    if not f or ver.prohibido(f) or ver.enunciado_trivial(f) or ver.clave(f) in claves:
                        continue
                    claves.add(ver.clave(f))
                    self.estado["contador"] += 1
                    slug = re.sub(r"[^A-Za-z0-9_]", "_", ver.nombre(f))[:30] or "lema"
                    firmas.append(ver.renombrar(f, f"l{self.estado['contador']}_{slug}"))
                if not firmas:
                    raise ValueError("no se obtuvo ningún lema útil y nuevo")
                codigo = ensamblar(ver.codigo(self.estado, probados), *[f"{f}\n  sorry" for f in firmas], f"{n['firma']}\n  sorry")
                ok, salida = ver.comprobar(codigo, permitir_sorry=True)
                if not ok:
                    raise ValueError(ver.resumir_error(salida))
                self.evento("plan", nodo=n["id"], lemas=firmas)
                self.say(n, f"plan con {len(firmas)} lemas: " + "; ".join(recortar(f.splitlines()[0], 60) for f in firmas))
                return [self.nodo(f, n["prof"] + 1) for f in firmas]
            except (ValueError, KeyError, TypeError) as e:
                error = str(e)
                self.evento("plan_fallo", nodo=n["id"], error=recortar(error, 300))
                self.say(n, f"plan {k}/{self.cfg.planes} rechazado: {recortar(error.splitlines()[0] if error else '', 120)}")
        return []

    def resolver_nodo(self, n: Dict[str, Any], ctx: List[str]) -> bool:
        if n["estado"] == "probado":
            return True
        ver = self.ver
        if not n["auto"]:
            self.presupuesto()
            cuerpo = ver.auto(self.estado, n["firma"], ctx)
            n["auto"] = True
            self.guardar()
            if cuerpo:
                return self._aceptar(n, cuerpo, "auto")
        while n["directos"] < self.cfg.directos:
            if self._intento(n, ctx, "directo"):
                return True
        if not ver.admite_plan or n["prof"] >= self.cfg.profundidad:
            return self._fallar(n)
        while n["replanes"] <= self.cfg.replanes:
            if n["necesita_plan"]:
                if n["replanes"] >= self.cfg.replanes:
                    break
                n["replanes"] += 1
                nuevos = self._planificar(n, ctx)
                n["hijos"].extend(nuevos)
                n["necesita_plan"] = not nuevos
                n["finales"] = 0
                self.guardar()
                if not nuevos:
                    continue
            hubo_fallo = False
            for i, h in enumerate(n["hijos"]):
                if h["estado"] == "probado":
                    continue
                ctx_h = ctx + [self.bloque_nodo(x) for x in n["hijos"][:i] if x["estado"] == "probado"]
                if not self.resolver_nodo(h, ctx_h):
                    n["fallidos"].append(h["firma"])
                    n["hijos"] = [x for x in n["hijos"] if x["estado"] == "probado"]   # se conservan los lemas ya demostrados
                    n["necesita_plan"] = True
                    hubo_fallo = True
                    self.guardar()
                    break
            if hubo_fallo:
                continue
            ctx_f = ctx + [self.bloque_nodo(x) for x in n["hijos"]]
            while n["finales"] < self.cfg.finales:
                if self._intento(n, ctx_f, "final"):
                    return True
            n["necesita_plan"] = True    # los lemas no bastaron: pedir más
            self.guardar()
        return self._fallar(n)

    def _fallar(self, n: Dict[str, Any]) -> bool:
        n["estado"] = "fallido"
        self.guardar()
        self.evento("fallido", nodo=n["id"], firma=recortar(n["firma"], 200))
        self.say(n, "no se pudo demostrar: " + recortar(n["firma"].splitlines()[0], 100))
        return False

    # ------------------------------------------------------------ cierre
    def finalizar(self) -> bool:
        ver, raiz = self.ver, self.estado["raiz"]
        codigo = ver.codigo(self.estado, [self.bloque_nodo(raiz)])
        nombre = ver.nombre(raiz["firma"]) if isinstance(ver, LeanVerificador) else "x"
        ok, salida = ver.comprobar(codigo, nombre_final=nombre)
        if not ok:
            self.estado["ultimo_error"] = recortar(salida, 800)
            self.guardar()
            self.say(None, "La verificación final FALLÓ: " + recortar(salida, 300))
            raiz["estado"] = "pendiente"   # se volverá a intentar en la próxima ejecución
            raiz["directos"] = raiz["finales"] = 0
            self.guardar()
            return False
        archivo = self.dir / f"resultado.{ver.extension}"
        escribir_atomico(archivo, codigo + "\n")
        self.estado["resultado"] = {"codigo": codigo, "archivo": str(archivo)}
        self.estado["estado"] = "verificado"
        self.guardar()
        self.evento("verificado", archivo=str(archivo))
        biblioteca_anadir(self.cfg.biblioteca, {"id": self.estado["problema"].get("id", ""), "backend": self.cfg.backend,
                          "enunciado": self.estado["problema"]["enunciado"], "codigo": codigo,
                          "etiquetas": list(self.estado["problema"].get("etiquetas", []))})
        return True

    def ejecutar(self, problema: Dict[str, Any], reiniciar: bool = False) -> int:
        self.say(None, f"Verificador: {self.ver.entorno()} | modelo {self.cfg.modelo}")
        self.cargar(problema, reiniciar)
        self.ver.ejemplos = biblioteca_ejemplos(self.cfg.biblioteca, problema["enunciado"], self.cfg.backend) if self.cfg.biblioteca else ""
        self.t_inicio = time.time()
        previo = signal.getsignal(signal.SIGTERM) if hasattr(signal, "SIGTERM") else None

        def al_terminar(signum: int, frame: Any) -> None:
            raise KeyboardInterrupt()
        try:
            if hasattr(signal, "SIGTERM"):
                signal.signal(signal.SIGTERM, al_terminar)
            if self.estado["estado"] == "verificado":
                self.say(None, "Este problema ya estaba verificado: " + self.estado["resultado"]["archivo"])
                return 0
            self.formalizar()
            if not self.resolver_nodo(self.estado["raiz"], []):
                self.estado["estado"] = "fallido"
                self.guardar()
                self.say(None, "No se encontró una demostración con los límites actuales. Sube --directos/--replanes/--profundidad y vuelve a ejecutar (con --reiniciar si quieres empezar de cero).")
                return 1
            if not self.finalizar():
                return 1
            self.say(None, "PROBLEMA RESUELTO Y VERIFICADO: " + self.estado["resultado"]["archivo"])
            return 0
        except Pausa as p:
            self.estado["estado"] = "pausado"
            self.guardar()
            self.say(None, f"PAUSADO ({p}). El progreso está guardado: ejecuta el mismo comando para continuar.")
            return 3
        except KeyboardInterrupt:
            self.estado["estado"] = "pausado"
            self.guardar()
            self.say(None, "INTERRUMPIDO. El progreso está guardado: ejecuta el mismo comando para continuar.")
            return 130
        except ErrorModelo as e:
            self.estado["estado"] = "pausado"
            self.guardar()
            self.say(None, f"ERROR DEL MODELO: {e}\nEl progreso está guardado: arranca Ollama y vuelve a ejecutar el mismo comando.")
            return 4
        except RuntimeError as e:
            self.estado["estado"] = "fallido"
            self.estado["error_fatal"] = str(e)      # ultimo_error conserva el error bruto del verificador
            self.guardar()
            self.say(None, f"ERROR: {e}")
            return 1
        finally:
            if previo is not None and hasattr(signal, "SIGTERM"):
                signal.signal(signal.SIGTERM, previo)


# =============================================================================
# 6. INFORME Y LÍNEA DE COMANDOS
# =============================================================================
def arbol(n: Dict[str, Any], nivel: int = 0) -> List[str]:
    marca = {"probado": "✔", "fallido": "✘", "pendiente": "…"}.get(n["estado"], "?")
    linea = f"{'    ' * nivel}{marca} {n['id']} [{n['estado']}] directos={n['directos']} finales={n['finales']} replanes={n['replanes']}  {recortar(n['firma'].splitlines()[0], 90)}"
    return [linea] + [l for h in n["hijos"] for l in arbol(h, nivel + 1)]


def informe(directorio: Path) -> str:
    ruta = Path(directorio) / "estado.json"
    if not ruta.exists():
        return f"No hay estado en {directorio}"
    e = json.loads(ruta.read_text(encoding="utf-8"))
    t = float(e.get("tiempo", 0))
    out = [f"Problema: {recortar(e['problema']['enunciado'].strip(), 150)}", f"Estado: {e['estado']} | backend {e['problema'].get('backend')} | "
           f"llamadas {e['llamadas']} | tiempo {t / 3600:.2f} h"]
    if e.get("raiz"):
        out += ["", *arbol(e["raiz"])]
    if e.get("error_fatal"):
        out += ["", "Error: " + recortar(e["error_fatal"], 300)]
    elif e.get("ultimo_error"):
        out += ["", "Último error: " + recortar(e["ultimo_error"], 300)]
    if e["estado"] == "verificado":
        out += ["", "Resultado: " + e["resultado"]["archivo"]]
    return "\n".join(out)


# Problemas de ejemplo INCLUIDOS en el programa (no hace falta ningún archivo): --problema par_mayor
ESPEC_ESCAPE: Dict[str, Any] = {
    "descripcion": "Deducir la velocidad de escape v_esc(r) con energía mecánica total cero y comprobar que en r_s = 2GM/c^2 vale c.",
    "preambulo": "G, M, c, r, m, v = symbols('G M c r m v', positive=True)\nr_s = 2*G*M/c**2",
    "variables": [
        "v_esc"
    ],
    "criterios": [
        "simplify(v_esc**2 - 2*G*M/r) == 0",
        "simplify(v_esc.subs(r, r_s) - c) == 0",
        "v_esc.is_positive == True"
    ]
}


PROBLEMAS_INCLUIDOS: Dict[str, Dict[str, Any]] = {
    "par_mayor": {
        "id": "par_mayor",
        "dominio": "matematicas",
        "backend": "lean",
        "enunciado": "Demuestra la existencia de un número natural m mayor que cualquier número natural n dado, tal que m sea además un número par."
    },
    "escape_schwarzschild": {
        "id": "escape_schwarzschild",
        "dominio": "fisica",
        "backend": "sympy",
        "enunciado": "Una masa m escapa de la atracción de una masa M desde una distancia r, justo con energía mecánica total cero (energía cinética + energía potencial gravitatoria -G M m / r). Deduce la velocidad de escape v_esc(r) y demuestra que a la distancia r_s = 2 G M / c^2 (el radio de Schwarzschild) la velocidad de escape vale exactamente c.",
        "especificacion": ESPEC_ESCAPE
    },
}
ALIAS_PROBLEMAS = {"facil_par_mayor": "par_mayor", "facil_velocidad_escape_schwarzschild": "escape_schwarzschild"}


def problema_incluido(nombre: str) -> Optional[Dict[str, str]]:
    """Busca un problema incluido por nombre o por la ruta de su archivo (p. ej. problemas/facil_par_mayor.json)."""
    clave = Path(nombre).stem if nombre.endswith(".json") else nombre
    clave = ALIAS_PROBLEMAS.get(clave, clave)
    p = PROBLEMAS_INCLUIDOS.get(clave)
    return dict(p) if p else None


def cargar_problema(a: argparse.Namespace) -> Dict[str, Any]:
    if a.problema:
        ruta = Path(a.problema)
        if ruta.is_file():
            p = json.loads(ruta.read_text(encoding="utf-8"))
            if "enunciado" not in p:
                raise SystemExit("El archivo del problema necesita el campo 'enunciado'.")
            p.setdefault("id", ruta.stem)
        else:
            p = problema_incluido(a.problema)
            if p is None:
                raise SystemExit(f"No existe el archivo de problema '{a.problema}'.\nProblemas incluidos (úsalos con --problema NOMBRE): "
                                 f"{', '.join(PROBLEMAS_INCLUIDOS)}.\nO pasa el enunciado directamente con --enunciado \"texto\".")
    elif a.enunciado:
        p = {"id": "", "enunciado": a.enunciado}
    else:
        raise SystemExit("Indica --problema ARCHIVO.json (o NOMBRE de un problema incluido) o --enunciado \"texto\".")
    p["enunciado"] = textwrap.dedent(p["enunciado"]).strip()
    p["backend"] = a.backend or p.get("backend", "lean")
    p.setdefault("dominio", "matematicas")
    if not p["id"]:
        p["id"] = re.sub(r"\W+", "_", p["enunciado"][:30]).strip("_").lower() or "problema"
    return p


def construir_config(a: argparse.Namespace, problema: Dict[str, Any]) -> Config:
    cfg = Config(modelo=a.modelo, ollama_url=a.ollama_url or os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434"), backend=problema["backend"],
                 lean=a.lean or "", lake_dir=a.lake_dir or "", mathlib=a.mathlib, horas=a.horas, max_intentos=a.max_intentos,
                 directos=a.directos, finales=a.finales, replanes=a.replanes, profundidad=a.profundidad, num_ctx=a.num_ctx,
                 num_predict=a.num_predict, fidelidad=not a.sin_fidelidad, semilla=a.semilla, timeout_lean=a.timeout_lean,
                 biblioteca=getattr(a, "biblioteca", "") or "")
    return cfg


def crear_verificador(cfg: Config) -> Verificador:
    if cfg.backend == "lean":
        return LeanVerificador(cfg)
    if cfg.backend == "sympy":
        return SympyVerificador(cfg)
    raise SystemExit(f"backend desconocido: {cfg.backend} (usa lean o sympy)")


def cmd_resolver(a: argparse.Namespace) -> int:
    problema = cargar_problema(a)
    cfg = construir_config(a, problema)
    ver = crear_verificador(cfg)
    base = Path(a.dir) if a.dir else Path("demostrador_runs") / f"{problema['id']}_{huella(problema['enunciado'] + problema['backend'])[:8]}"
    llm = LLM(cfg)
    motor = Motor(cfg, llm, ver, base)
    print(f"Directorio de trabajo (estado y resultado): {base}", flush=True)
    try:
        return motor.ejecutar(problema, a.reiniciar)
    except RuntimeError as e:
        print(f"ERROR: {e}")
        return 2


def cmd_curriculo(a: argparse.Namespace) -> int:
    """Resuelve en orden los problemas de un currículo de dificultad creciente; cada uno verificado alimenta la biblioteca."""
    if not Path(a.archivo).is_file():
        raise SystemExit(f"No existe el currículo '{a.archivo}'. En Colab ejecuta antes la celda que lo escribe (sección 8) o pasa --archivo RUTA.")
    datos = json.loads(Path(a.archivo).read_text(encoding="utf-8"))
    lista = list(datos.get(a.pista, []))
    if not lista:
        raise SystemExit(f"El currículo no tiene la pista '{a.pista}' (usa lean, sympy o abiertos).")
    lista.sort(key=lambda p: p.get("nivel", 0))
    base_dir = Path(a.dir)
    prog_path = base_dir / f"progreso_{a.pista}.json"
    prog = json.loads(prog_path.read_text(encoding="utf-8")) if prog_path.is_file() else {}
    a.biblioteca = a.biblioteca or str(base_dir / f"biblioteca_{a.pista}.json")
    resueltos = 0
    for p in lista:
        if a.desde and p.get("nivel", 0) < a.desde:
            continue
        if a.hasta and p.get("nivel", 0) > a.hasta:
            break
        if prog.get(p["id"]) == "verificado":
            print(f"[nivel {p.get('nivel')}] {p['id']}: ya verificado", flush=True)
            resueltos += 1
            continue
        print(f"\n===== NIVEL {p.get('nivel')}: {p.get('titulo', p['id'])} =====", flush=True)
        enun = p["enunciado"] + (("\nPista de estrategia: " + p["idea"]) if p.get("idea") and not a.sin_pistas else "")
        a.problema, a.enunciado = None, None
        pr = {"id": p["id"], "enunciado": enun, "backend": a.pista if a.pista in ("lean", "sympy") else p.get("backend", "lean"),
              "dominio": "curriculo", "etiquetas": p.get("etiquetas", [])}
        if p.get("especificacion"):
            pr["especificacion"] = p["especificacion"]
        cfg = construir_config(a, pr)
        motor = Motor(cfg, LLM(cfg), crear_verificador(cfg), base_dir / "runs" / p["id"])
        try:
            codigo = motor.ejecutar(pr, False)
        except RuntimeError as e:
            print(f"ERROR: {e}")
            return 2
        prog[p["id"]] = {0: "verificado", 1: "fallido"}.get(codigo, "pausado")
        escribir_atomico(prog_path, json.dumps(prog, ensure_ascii=False, indent=1))
        if codigo == 0:
            resueltos += 1
        elif codigo != 1 or not a.seguir:
            print(f"\nCurrículo detenido en el nivel {p.get('nivel')} (código {codigo}). Vuelve a ejecutar el mismo comando para continuar.")
            return codigo
    print(f"\nCurrículo '{a.pista}': {resueltos}/{len(lista)} verificados. Biblioteca: {a.biblioteca}")
    return 0


def construir_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog="demostrador.py", description="Demostrador de problemas matemáticos y físicos con LLM local y verificador externo.")
    sub = p.add_subparsers(dest="cmd")
    r = sub.add_parser("resolver", help="resolver (o reanudar) un problema")
    r.add_argument("--problema", help="archivo JSON con enunciado, backend y dominio")
    r.add_argument("--enunciado", help="enunciado en texto")
    r.add_argument("--backend", choices=["lean", "sympy"], default=None)
    r.add_argument("--dir", help="carpeta de estado/resultado (por defecto demostrador_runs/<id>_<hash>)")
    r.add_argument("--modelo", default=MODELO_DEFECTO)
    r.add_argument("--ollama-url", default=None)
    r.add_argument("--lean", default="", help="ruta al ejecutable lean")
    r.add_argument("--lake-dir", default="", help="proyecto lake con Mathlib (usa `lake env lean`)")
    r.add_argument("--mathlib", action="store_true", help="permitir tácticas y tipos de Mathlib (requiere --lake-dir)")
    r.add_argument("--horas", type=float, default=0.0, help="límite de tiempo de ESTA sesión (0 = sin límite)")
    r.add_argument("--max-intentos", type=int, default=0, help="límite de llamadas al modelo en esta sesión (0 = sin límite)")
    r.add_argument("--directos", type=int, default=6)
    r.add_argument("--finales", type=int, default=6)
    r.add_argument("--replanes", type=int, default=3)
    r.add_argument("--profundidad", type=int, default=2)
    r.add_argument("--num-ctx", type=int, default=8192)
    r.add_argument("--num-predict", type=int, default=1000, help="tokens máximos por respuesta (en CPU, 500-700 acelera mucho)")
    r.add_argument("--timeout-lean", type=int, default=90)
    r.add_argument("--semilla", type=int, default=1)
    r.add_argument("--sin-fidelidad", action="store_true", help="no comprobar con el modelo que el enunciado formal es fiel")
    r.add_argument("--reiniciar", action="store_true", help="descartar el estado guardado y empezar de cero")
    r.add_argument("--biblioteca", default="", help="JSON de demostraciones verificadas: se usan como ejemplos y se amplía al resolver")
    c = sub.add_parser("curriculo", help="resolver en orden un currículo de problemas cada vez más difíciles")
    c.add_argument("--archivo", default="curriculo/curriculo.json")
    c.add_argument("--pista", choices=["lean", "sympy", "abiertos"], default="lean")
    c.add_argument("--dir", default="curriculo_runs", help="carpeta de progreso, estados y biblioteca (usa Drive en Colab)")
    c.add_argument("--desde", type=int, default=0)
    c.add_argument("--hasta", type=int, default=0)
    c.add_argument("--seguir", action="store_true", help="si un nivel no se resuelve, seguir con el siguiente")
    c.add_argument("--sin-pistas", action="store_true", help="no añadir la 'idea' del currículo al enunciado")
    c.add_argument("--biblioteca", default="")
    for k, d in (("modelo", MODELO_DEFECTO), ("ollama_url", None), ("lean", ""), ("lake_dir", "")):
        c.add_argument("--" + k.replace("_", "-"), default=d)
    c.add_argument("--mathlib", action="store_true")
    for k, d in (("horas", 0.0), ("max_intentos", 0), ("directos", 6), ("finales", 6), ("replanes", 3), ("profundidad", 2),
                 ("num_ctx", 8192), ("num_predict", 700), ("timeout_lean", 90), ("semilla", 1)):
        c.add_argument("--" + k.replace("_", "-"), type=type(d), default=d)
    c.add_argument("--sin-fidelidad", action="store_true")
    sub.add_parser("problemas", help="listar los problemas de ejemplo incluidos")
    e = sub.add_parser("estado", help="mostrar el progreso de una ejecución")
    e.add_argument("--dir", required=True)
    sub.add_parser("selftest", help="ejecutar las pruebas (tests/test_demostrador.py)")
    return p


def main(argv: Optional[List[str]] = None) -> int:
    for s in (sys.stdout, sys.stderr):
        try:
            s.reconfigure(encoding="utf-8", errors="replace")  # type: ignore[attr-defined]
        except Exception:  # noqa: BLE001
            pass
    a = construir_parser().parse_args(argv)
    if a.cmd == "resolver":
        return cmd_resolver(a)
    if a.cmd == "curriculo":
        return cmd_curriculo(a)
    if a.cmd == "problemas":
        for k, v in PROBLEMAS_INCLUIDOS.items():
            print(f"{k}  [{v['backend']}, {v['dominio']}]  {v['enunciado'][:110]}")
        return 0
    if a.cmd == "estado":
        print(informe(Path(a.dir)))
        return 0
    if a.cmd == "selftest":
        import unittest
        suite = unittest.defaultTestLoader.discover(str(Path(__file__).resolve().parent / "tests"), pattern="test_demostrador.py")
        return 0 if unittest.TextTestRunner(verbosity=1).run(suite).wasSuccessful() else 1
    construir_parser().print_help()
    return 2


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
!python demostrador.py problemas   # lista los problemas de ejemplo (van dentro del programa, no hacen falta archivos)

## 4. Problema fácil (matemáticas, Lean 4)

*«Demuestra la existencia de un número natural m mayor que cualquier n dado, tal que m sea par.»*

- Si falla o lo interrumpes, **vuelve a ejecutar la celda**: reintenta con los contadores a cero y conserva lo ya demostrado.
- Para empezar de cero, añade `--reiniciar`.

In [ ]:
!python demostrador.py resolver --problema par_mayor --modelo {MODELO} --lean {LEAN} --dir {DIR}/par_mayor

## 5. Problema fácil de física (SymPy): velocidad de escape y radio de Schwarzschild

In [ ]:
!python demostrador.py resolver --problema escape_schwarzschild --modelo {MODELO} --dir {DIR}/escape

> En física, los **criterios** de verificación los escribe el modelo antes de resolver (salen en el registro). **Revísalos**: la verificación es tan fiable como esos criterios.

## 6. Ver el progreso y el resultado

In [ ]:
!python demostrador.py estado --dir {DIR}/par_mayor
print()
!python demostrador.py estado --dir {DIR}/escape

In [ ]:
print(open(f'{DIR}/par_mayor/resultado.lean').read())

## 7. Problemas largos o difíciles

- Dale **horas**: `--horas 3` (límite de la sesión; al llegar se pausa con el progreso guardado).
- Más intentos y descomposición más profunda: `--directos 10 --replanes 5 --profundidad 3`.
- Un modelo mayor ayuda mucho: `MODELO = 'qwen2.5:14b'` (si la GPU lo aguanta).
- **Sin GPU (solo CPU)** cada llamada de un 7B tarda minutos. Para acelerar: `--num-predict 600 --num-ctx 4096` (respuestas más cortas) y/o un modelo más pequeño (`qwen2.5:3b`). Lo ideal es esperar a que se renueve la cuota de GPU: el progreso queda guardado en Drive y se reanuda.
- En segundo plano, consultando el registro:

In [ ]:
!nohup python demostrador.py resolver --problema par_mayor --modelo {MODELO} --lean {LEAN} --dir {DIR}/par_mayor --horas 3 > {DIR}/par_mayor.log 2>&1 &

In [ ]:
!tail -n 20 {DIR}/par_mayor.log

Los problemas con **Mathlib** (números reales, análisis...) necesitan un proyecto *lake* con Mathlib (descarga grande); después: `--lake-dir RUTA --mathlib`.
Para problemas nuevos (agujeros negros, espacio-tiempo) usa `--enunciado "texto" --backend lean|sympy`, o un archivo `.json` con `enunciado`, `backend` y `dominio` (`--problema ruta.json`).

## 8. Currículo: demostraciones cada vez más difíciles (la IA «aprende» sin cambiar de modelo)

Cómo se automejora un modelo pequeño **sin tocar sus pesos**:
1. Resuelve en orden **18 problemas de Lean** (de `omega` a inducción, definiciones recursivas, mcd y Fibonacci) y **12 de física** (de la velocidad de escape a *Schwarzschild resuelve las ecuaciones de vacío*).
2. Cada demostración **verificada** entra en una **biblioteca** (en Drive). En los problemas siguientes, las más parecidas se le enseñan al modelo como ejemplos: el nivel *n* se apoya en los niveles anteriores.
3. Si un nivel no sale, el currículo se detiene ahí (`--seguir` para saltarlo). Vuelve a ejecutar la celda: **reanuda donde estaba**.

Por qué es honesto: lo verificado lo decide Lean/SymPy, no el modelo; el currículo mide hasta dónde llega *este* modelo con *este* hardware. No garantiza resolver problemas abiertos: los de la sección «abiertos» se tratan solo en su **versión acotada y comprobable**; el enunciado general sigue abierto.

In [ ]:
import os
os.makedirs('curriculo', exist_ok=True)
open('curriculo/curriculo.json', 'w', encoding='utf-8').write('{\n  "version": 1,\n  "lean": [\n    {\n      "nivel": 1,\n      "id": "cota_suma_lineal",\n      "titulo": "Una cota a partir de una suma",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todos los números naturales a y b, si a + b = 10 entonces a ≤ 10 y b ≤ 10. Formalización: `theorem cota_suma_lineal (a b : Nat) (h : a + b = 10) : a ≤ 10 ∧ b ≤ 10`.",\n      "idea": "Es aritmética lineal pura sobre Nat: separa la conjunción con `constructor` y cierra cada parte con `omega`, o cierra todo directamente con `omega`.",\n      "etiquetas": [\n        "omega",\n        "conjuncion",\n        "aritmetica_lineal"\n      ]\n    },\n    {\n      "nivel": 2,\n      "id": "impar_mas_uno_es_par",\n      "titulo": "Sucesor de un impar",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todo natural n, si n % 2 = 1 entonces (n + 1) % 2 = 0. Formalización: `theorem impar_mas_uno_es_par (n : Nat) (h : n % 2 = 1) : (n + 1) % 2 = 0`.",\n      "idea": "Introduce la hipótesis y usa `omega`, que entiende el módulo por una constante literal.",\n      "etiquetas": [\n        "omega",\n        "implicacion",\n        "paridad",\n        "modulo"\n      ]\n    },\n    {\n      "nivel": 3,\n      "id": "existe_multiplo_de_tres_mayor",\n      "titulo": "Existe un múltiplo de 3 mayor que n",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todo natural n existe un natural m tal que m > n y m % 3 = 0. Formalización: `theorem existe_multiplo_de_tres_mayor (n : Nat) : ∃ m : Nat, m > n ∧ m % 3 = 0`.",\n      "idea": "Da el testigo explícito `3 * (n + 1)` con `exact ⟨3 * (n + 1), by omega, by omega⟩` (o `refine` seguido de `constructor`).",\n      "etiquetas": [\n        "existencial",\n        "testigo",\n        "omega",\n        "modulo"\n      ]\n    },\n    {\n      "nivel": 4,\n      "id": "multiplo_de_seis_implica_dos_y_tres",\n      "titulo": "Múltiplos de 6",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todo natural n, si n % 6 = 0 entonces n % 2 = 0 y n % 3 = 0. Formalización: `theorem multiplo_de_seis (n : Nat) (h : n % 6 = 0) : n % 2 = 0 ∧ n % 3 = 0`.",\n      "idea": "Separa la conjunción con `constructor` y resuelve cada objetivo con `omega` usando la hipótesis h.",\n      "etiquetas": [\n        "omega",\n        "conjuncion",\n        "divisibilidad",\n        "modulo"\n      ]\n    },\n    {\n      "nivel": 5,\n      "id": "suma_de_dos_impares",\n      "titulo": "Suma de dos impares",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todos los naturales a y b, si a % 2 = 1 y b % 2 = 1 entonces (a + b) % 2 = 0 y además (a + b + 1) % 2 = 1. Formalización: `theorem suma_de_dos_impares (a b : Nat) (ha : a % 2 = 1) (hb : b % 2 = 1) : (a + b) % 2 = 0 ∧ (a + b + 1) % 2 = 1`.",\n      "idea": "Dos variables y dos hipótesis, todo lineal con módulo literal: `constructor <;> omega`.",\n      "etiquetas": [\n        "omega",\n        "conjuncion",\n        "paridad",\n        "dos_variables"\n      ]\n    },\n    {\n      "nivel": 6,\n      "id": "cota_division_entera_por_dos",\n      "titulo": "Cotas de la división entera por 2",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todo natural n se cumple 2 * (n / 2) ≤ n y n < 2 * (n / 2) + 2. Formalización: `theorem cota_division (n : Nat) : 2 * (n / 2) ≤ n ∧ n < 2 * (n / 2) + 2`.",\n      "idea": "`omega` soporta la división entera por constantes literales; basta `constructor <;> omega`.",\n      "etiquetas": [\n        "omega",\n        "division_entera",\n        "desigualdades"\n      ]\n    },\n    {\n      "nivel": 7,\n      "id": "menor_o_igual_implica_diferencia",\n      "titulo": "Desigualdad y existencia de la diferencia",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todos los naturales a y b, si a ≤ b entonces existe un natural c con b = a + c, y que si además a < b entonces existe un natural d con b = a + d + 1. Formalización: `theorem diferencia (a b : Nat) (h : a ≤ b) : (∃ c : Nat, b = a + c) ∧ (a < b → ∃ d : Nat, b = a + d + 1)`.",\n      "idea": "Construye los testigos `b - a` y `b - a - 1` y cierra las igualdades con `omega`; para la segunda parte introduce la hipótesis `a < b` con `intro`.",\n      "etiquetas": [\n        "existencial",\n        "testigo",\n        "implicacion",\n        "omega",\n        "desigualdades"\n      ]\n    },\n    {\n      "nivel": 8,\n      "id": "producto_de_impares_es_impar",\n      "titulo": "El producto de dos impares es impar",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todos los naturales a y b, si a % 2 = 1 y b % 2 = 1 entonces (a * b) % 2 = 1. Formalización: `theorem producto_impares (a b : Nat) (ha : a % 2 = 1) (hb : b % 2 = 1) : (a * b) % 2 = 1`.",\n      "idea": "El producto no es lineal, así que `omega` no basta: reescribe con `Nat.mul_mod`, sustituye ha y hb y cierra con `simp` o `norm_num`/`decide` sobre el resultado numérico.",\n      "etiquetas": [\n        "modulo",\n        "Nat.mul_mod",\n        "rw",\n        "simp",\n        "paridad"\n      ]\n    },\n    {\n      "nivel": 9,\n      "id": "cuadrado_modulo_cuatro",\n      "titulo": "Cuadrados módulo 4",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todo natural n, (n * n) % 4 = 0 o (n * n) % 4 = 1. Formalización: `theorem cuadrado_mod_cuatro (n : Nat) : (n * n) % 4 = 0 ∨ (n * n) % 4 = 1`.",\n      "idea": "Distingue los cuatro casos de n % 4 (con `omega` obtén `n % 4 = 0 ∨ ... ∨ n % 4 = 3` y `rcases`), reescribe con `Nat.mul_mod` y calcula con `simp [h]`.",\n      "etiquetas": [\n        "casos",\n        "rcases",\n        "modulo",\n        "Nat.mul_mod",\n        "disyuncion"\n      ]\n    },\n    {\n      "nivel": 10,\n      "id": "suma_de_impares_es_cuadrado",\n      "titulo": "La suma de los n primeros impares es n²",\n      "enunciado": "En Lean 4 (sin Mathlib) define la función recursiva `def sumaImpares : Nat → Nat | 0 => 0 | n + 1 => sumaImpares n + (2 * n + 1)` y demuestra que para todo natural n, `sumaImpares n = n * n`.",\n      "idea": "Inducción simple sobre n: el caso base es `rfl`; en el paso desarrolla con `simp [sumaImpares, ih]`, expande `(n + 1) * (n + 1)` con `Nat.add_mul`/`Nat.mul_add` (o `Nat.succ_mul`) y termina con `omega`.",\n      "etiquetas": [\n        "induccion",\n        "def_recursiva",\n        "simp",\n        "nlinealidad"\n      ]\n    },\n    {\n      "nivel": 11,\n      "id": "n_menor_que_dos_pot_n",\n      "titulo": "n < 2^n",\n      "enunciado": "Demuestra en Lean 4 (sin Mathlib) que para todo natural n se cumple n < 2 ^ n. Formalización: `theorem lt_two_pow (n : Nat) : n < 2 ^ n`.",\n      "idea": "Inducción sobre n; en el paso reescribe `2 ^ (n + 1) = 2 ^ n * 2` con `Nat.pow_succ` y cierra con `omega` usando la hipótesis de inducción (2 ^ n se trata como átomo).",\n      "etiquetas": [\n        "induccion",\n        "potencias",\n        "Nat.pow_succ",\n        "omega"\n      ]\n    },\n    {\n      "nivel": 12,\n      "id": "suma_de_gauss_recursiva",\n      "titulo": "Fórmula de Gauss con suma recursiva",\n      "enunciado": "En Lean 4 (sin Mathlib) define `def suma : Nat → Nat | 0 => 0 | n + 1 => suma n + (n + 1)` (la suma 1 + 2 + ... + n) y demuestra que para todo natural n, `2 * suma n = n * (n + 1)`.",\n      "idea": "Inducción sobre n; en el paso despliega `suma (n + 1)`, usa la hipótesis de inducción y expande el producto `(n + 1) * (n + 2)` con `Nat.mul_add`/`Nat.add_mul` antes de `omega`.",\n      "etiquetas": [\n        "induccion",\n        "def_recursiva",\n        "suma",\n        "nlinealidad"\n      ]\n    },\n    {\n      "nivel": 13,\n      "id": "factorial_positivo_y_cota",\n      "titulo": "Factorial: positividad y cota inferior",\n      "enunciado": "En Lean 4 (sin Mathlib) define `def fact : Nat → Nat | 0 => 1 | n + 1 => (n + 1) * fact n` y demuestra, para todo natural n, las dos propiedades `0 < fact n` y `n ≤ fact n`. Se recomienda probar primero la positividad como lema auxiliar y usarla en la segunda.",\n      "idea": "Lema 1 por inducción con `Nat.mul_pos`; lema 2 por inducción usando que `n + 1 ≤ (n + 1) * fact n` (por ejemplo con `Nat.le_mul_of_pos_right` y el lema 1).",\n      "etiquetas": [\n        "induccion",\n        "def_recursiva",\n        "lema_auxiliar",\n        "Nat.mul_pos",\n        "desigualdades"\n      ]\n    },\n    {\n      "nivel": 14,\n      "id": "suma_recursiva_correcta_y_conmutativa",\n      "titulo": "Suma recursiva: corrección y conmutatividad",\n      "enunciado": "En Lean 4 (sin Mathlib) define `def sumaR : Nat → Nat → Nat | a, 0 => a | a, b + 1 => sumaR a b + 1`. Demuestra (1) para todos a y b, `sumaR a b = a + b`; y (2) para todos a y b, `sumaR a b = sumaR b a`.",\n      "idea": "La propiedad (1) sale por inducción sobre b con `simp [sumaR, ih]`; (2) se deduce rápidamente reescribiendo con (1) y usando `Nat.add_comm` u `omega`.",\n      "etiquetas": [\n        "induccion",\n        "def_recursiva",\n        "dos_variables",\n        "lema_auxiliar",\n        "conmutatividad"\n      ]\n    },\n    {\n      "nivel": 15,\n      "id": "suma_de_potencias_de_dos",\n      "titulo": "Suma de potencias de 2",\n      "enunciado": "En Lean 4 (sin Mathlib) define `def sp : Nat → Nat | 0 => 1 | n + 1 => sp n + 2 ^ (n + 1)` (es 1 + 2 + 4 + ... + 2^n) y demuestra que para todo natural n, `sp n + 1 = 2 ^ (n + 1)`.",\n      "idea": "Inducción sobre n; en el paso despliega `sp (n + 1)`, reescribe `2 ^ (n + 2)` y `2 ^ (n + 1)` con `Nat.pow_succ` para que `omega` cierre tratando `2 ^ (n + 1)` como átomo.",\n      "etiquetas": [\n        "induccion",\n        "def_recursiva",\n        "potencias",\n        "Nat.pow_succ",\n        "omega"\n      ]\n    },\n    {\n      "nivel": 16,\n      "id": "fibonacci_acotado_por_potencia_de_dos",\n      "titulo": "Fibonacci ≤ 2^n",\n      "enunciado": "En Lean 4 (sin Mathlib) define `def fib : Nat → Nat | 0 => 0 | 1 => 1 | n + 2 => fib (n + 1) + fib n` y demuestra que para todo natural n, `fib n ≤ 2 ^ n`.",\n      "idea": "Inducción fuerte (o inducción doble sobre el par n, n+1): casos n = 0, n = 1 y paso n + 2 usando las dos hipótesis y `2 ^ (n + 2) = 2 ^ n * 4` (`Nat.pow_succ`), cerrando con `omega`.",\n      "etiquetas": [\n        "induccion_fuerte",\n        "def_recursiva",\n        "fibonacci",\n        "potencias",\n        "omega"\n      ]\n    },\n    {\n      "nivel": 17,\n      "id": "mcd_euclides_divide",\n      "titulo": "El mcd de Euclides divide a ambos números",\n      "enunciado": "En Lean 4 (sin Mathlib) define `def mcd (a b : Nat) : Nat := if h : b = 0 then a else mcd b (a % b) termination_by b decreasing_by exact Nat.mod_lt _ (Nat.pos_of_ne_zero h)`. Demuestra que para todos a y b, `a % mcd a b = 0 ∧ b % mcd a b = 0`. (Convención de Nat: n % 0 = n, de modo que el caso a = b = 0 también se cumple.)",\n      "idea": "Inducción fuerte sobre b (o `induction b using Nat.strongRecOn` generalizando a): si b = 0 usa `mcd a 0 = a` y `Nat.mod_self`; si no, aplica la hipótesis a (b, a % b) y reconstruye a con `Nat.mod_add_div`, probando antes un lema de divisibilidad con `%`.",\n      "etiquetas": [\n        "induccion_fuerte",\n        "def_recursiva",\n        "mcd",\n        "divisibilidad",\n        "lemas_encadenados"\n      ]\n    },\n    {\n      "nivel": 18,\n      "id": "fibonacci_consecutivos_coprimos",\n      "titulo": "Fibonacci consecutivos son coprimos",\n      "enunciado": "En Lean 4 (sin Mathlib) con `fib` y `mcd` definidos como antes (fib 0 = 0, fib 1 = 1, fib (n + 2) = fib (n + 1) + fib n; mcd por el algoritmo de Euclides con `%`), demuestra que para todo natural n, `mcd (fib (n + 1)) (fib n) = 1`. Se sugiere probar antes los lemas auxiliares `mcd a 0 = a`, `mcd a b = mcd b (a % b)` para b ≠ 0 y `mcd (a + b) b = mcd a b`.",\n      "idea": "Inducción sobre n con el paso `mcd (fib (n + 2)) (fib (n + 1)) = mcd (fib (n + 1) + fib n) (fib (n + 1)) = mcd (fib n) (fib (n + 1))`; hace falta además la simetría `mcd a b = mcd b a` (por inducción fuerte) para enlazar con la hipótesis, o bien reformular el invariante con ambos órdenes a la vez.",\n      "etiquetas": [\n        "induccion",\n        "def_recursiva",\n        "fibonacci",\n        "mcd",\n        "lemas_encadenados",\n        "dos_variables"\n      ]\n    }\n  ],\n  "sympy": [\n    {\n      "nivel": 1,\n      "id": "velocidad_de_escape_newton",\n      "titulo": "Velocidad de escape newtoniana",\n      "enunciado": "Una masa de prueba m parte de la distancia r > 0 del centro de un cuerpo esférico de masa M, con velocidad v hacia fuera. Hipótesis: gravedad newtoniana, energía mecánica conservada y escape cuando la energía total vale 0 (llega al infinito con velocidad nula). Deriva con SymPy que v_esc = sqrt(2*G*M/r), partiendo de la ecuación (1/2)*m*v**2 - G*M*m/r = 0. Criterio: v_esc**2 - 2*G*M/r se simplifica a 0 y v_esc > 0 para G, M, r positivos.",\n      "idea": "Plantea la energía total como expresión simbólica, resuelve para v con `solve` (o `sympy.solve`) eligiendo la raíz positiva y simplifica.",\n      "etiquetas": [\n        "energia",\n        "solve",\n        "newton"\n      ],\n      "especificacion": {\n        "descripcion": "Deducir v_esc = sqrt(2GM/r) igualando a cero la energía total.",\n        "preambulo": "G, M, r, m, v = symbols(\'G M r m v\', positive=True)",\n        "variables": [\n          "v_esc"\n        ],\n        "criterios": [\n          "simplify(v_esc**2 - 2*G*M/r) == 0",\n          "v_esc.is_positive == True"\n        ]\n      }\n    },\n    {\n      "nivel": 2,\n      "id": "orbita_circular_newton",\n      "titulo": "Órbita circular y periodo",\n      "enunciado": "Una masa pequeña describe una órbita circular de radio r alrededor de una masa M (gravedad newtoniana, M >> m). Igualando la fuerza gravitatoria G*M*m/r**2 con la fuerza centrípeta m*v**2/r, deriva la velocidad orbital v = sqrt(G*M/r) y el periodo T = 2*pi*r/v = 2*pi*sqrt(r**3/(G*M)). Criterio: v**2 - G*M/r y T**2 - 4*pi**2*r**3/(G*M) se simplifican a 0.",\n      "idea": "Resuelve la igualdad de fuerzas para v con `solve`, define T = 2*pi*r/v y comprueba con `simplify` que T**2 coincide con la expresión pedida.",\n      "etiquetas": [\n        "orbitas",\n        "solve",\n        "newton"\n      ],\n      "especificacion": {\n        "descripcion": "Deducir la velocidad orbital v_orb y el periodo T de una órbita circular.",\n        "preambulo": "G, M, r, m, v = symbols(\'G M r m v\', positive=True)",\n        "variables": [\n          "v_orb",\n          "T"\n        ],\n        "criterios": [\n          "simplify(v_orb**2 - G*M/r) == 0",\n          "simplify(T**2 - 4*pi**2*r**3/(G*M)) == 0"\n        ]\n      }\n    },\n    {\n      "nivel": 3,\n      "id": "tercera_ley_de_kepler",\n      "titulo": "Tercera ley de Kepler",\n      "enunciado": "Para órbitas circulares de radio a alrededor de una masa M (gravedad newtoniana), demuestra simbólicamente que T**2 = 4*pi**2*a**3/(G*M), y por tanto que el cociente T**2/a**3 no depende de a (su derivada respecto de a es 0). Criterio: diff(T**2/a**3, a) se simplifica a 0 y T**2*G*M/(4*pi**2*a**3) se simplifica a 1.",\n      "idea": "Parte de la velocidad orbital del problema anterior, construye T y verifica la derivada respecto de a con `diff` y `simplify`.",\n      "etiquetas": [\n        "kepler",\n        "diff",\n        "simplify"\n      ],\n      "especificacion": {\n        "descripcion": "Deducir T2 = T**2 en función de a y comprobar que T2/a**3 no depende de a.",\n        "preambulo": "G, M, a, v = symbols(\'G M a v\', positive=True)",\n        "variables": [\n          "T2"\n        ],\n        "criterios": [\n          "simplify(T2 - 4*pi**2*a**3/(G*M)) == 0",\n          "simplify(diff(T2/a**3, a)) == 0"\n        ]\n      }\n    },\n    {\n      "nivel": 4,\n      "id": "fuerza_de_marea_newton",\n      "titulo": "Fuerza de marea",\n      "enunciado": "Un cuerpo de masa m y longitud pequeña L está orientado radialmente a distancia r de una masa M (gravedad newtoniana). La fuerza gravitatoria es F(r) = G*M*m/r**2. Deriva la diferencia de fuerza entre sus extremos en el límite L << r: la aceleración de marea es a_marea = 2*G*M*L/r**3 (en módulo). Criterio: derivada de F respecto de r, evaluada en r y multiplicada por -L, es igual a 2*G*M*m*L/r**3 (expresión simbólica, simplificada); y el desarrollo de Taylor de F(r+L) - F(r) a primer orden en L da -2*G*M*m*L/r**3 (signo: la fuerza disminuye al alejarse).",\n      "idea": "Deriva F con `diff` o usa `series` de F(r+L) en L hasta primer orden y compara con la expresión objetivo.",\n      "etiquetas": [\n        "mareas",\n        "diff",\n        "series"\n      ]\n    },\n    {\n      "nivel": 5,\n      "id": "radio_de_schwarzschild",\n      "titulo": "Radio de Schwarzschild",\n      "enunciado": "Hipótesis: se impone formalmente (argumento heurístico, que coincide con el resultado exacto de la relatividad general) que la velocidad de escape newtoniana v = sqrt(2*G*M/r) iguale la velocidad de la luz c. Deriva r_s = 2*G*M/c**2. Después calcula numéricamente r_s para M = 1 masa solar (M_sol = 1.989e30 kg, G = 6.674e-11 m**3/(kg*s**2), c = 2.998e8 m/s) y comprueba que está entre 2.9e3 m y 3.0e3 m. Criterio: r_s - 2*G*M/c**2 se simplifica a 0 y el valor numérico está en el intervalo indicado.",\n      "idea": "Resuelve sqrt(2*G*M/r) = c para r con `solve` y sustituye valores con `subs`/`evalf`.",\n      "etiquetas": [\n        "schwarzschild",\n        "solve",\n        "numerico"\n      ],\n      "especificacion": {\n        "descripcion": "Deducir r_s igualando la velocidad de escape a c.",\n        "preambulo": "G, M, c, r = symbols(\'G M c r\', positive=True)",\n        "variables": [\n          "r_s"\n        ],\n        "criterios": [\n          "simplify(r_s - 2*G*M/c**2) == 0"\n        ]\n      }\n    },\n    {\n      "nivel": 6,\n      "id": "dilatacion_temporal_schwarzschild",\n      "titulo": "Dilatación gravitatoria del tiempo y corrimiento al rojo",\n      "enunciado": "En la métrica de Schwarzschild (G = c = 1 o con r_s = 2*G*M/c**2), un observador estático a radio r > r_s tiene tiempo propio dtau = sqrt(1 - r_s/r)*dt, donde t es el tiempo del observador en el infinito. Un emisor estático en r emite luz de frecuencia nu_e y un observador en el infinito la recibe con nu_inf = nu_e*sqrt(1 - r_s/r). Deriva el corrimiento al rojo z = 1/sqrt(1 - r_s/r) - 1 y demuestra que, para r >> r_s, z ≈ r_s/(2*r) = G*M/(r*c**2) (primer orden). Criterio: series(z, r_s, 0, 2) tiene término lineal r_s/(2*r), y z vale 0 en el límite r -> oo.",\n      "idea": "Define z como expresión en r_s y r, expándela con `series` en r_s alrededor de 0 y calcula `limit` cuando r tiende a infinito.",\n      "etiquetas": [\n        "redshift",\n        "schwarzschild",\n        "series",\n        "limit"\n      ]\n    },\n    {\n      "nivel": 7,\n      "id": "potencial_efectivo_schwarzschild",\n      "titulo": "Potencial efectivo de Schwarzschild y órbita circular",\n      "enunciado": "Para una partícula masiva en geodésica ecuatorial de Schwarzschild, con unidades G = c = 1 y masa central M, la ecuación radial por unidad de masa es E**2 = (dr/dtau)**2 + V(r), con V(r) = (1 - 2*M/r)*(1 + L**2/r**2), donde E y L son la energía y el momento angular específicos conservados. Deriva las condiciones de órbita circular dV/dr = 0 y demuestra que L**2 = M*r**2/(r - 3*M) y E**2 = (1 - 2*M/r)**2/(1 - 3*M/r) para r > 3*M. Criterio: diff(V, r) se anula al sustituir L**2 por la expresión dada, y V evaluado ahí coincide con E**2.",\n      "idea": "Deriva V con `diff`, resuelve dV/dr = 0 para L**2 con `solve`, y sustituye para obtener E**2 simplificando.",\n      "etiquetas": [\n        "potencial_efectivo",\n        "geodesicas",\n        "solve",\n        "diff"\n      ]\n    },\n    {\n      "nivel": 8,\n      "id": "esfera_de_fotones",\n      "titulo": "Esfera de fotones: r = 3GM/c²",\n      "enunciado": "Para un rayo de luz (masa nula) en órbita circular ecuatorial de Schwarzschild, con G = c = 1 el potencial efectivo por unidad de L**2 es W(r) = (1 - 2*M/r)/r**2. Deriva que la órbita circular exige dW/dr = 0 y que la única solución con r > 0 es r = 3*M (en unidades con constantes: r = 3*G*M/c**2). Demuestra además que ahí W tiene un máximo (d2W/dr2 < 0 en r = 3*M), es decir, la órbita es inestable. Criterio: solve(diff(W, r), r) devuelve [3*M] para r > 0 y diff(W, r, 2) evaluada en r = 3*M es negativa para M > 0.",\n      "idea": "Usa `diff` y `solve` sobre W(r), y evalúa la segunda derivada en la raíz con `subs` y `simplify`.",\n      "etiquetas": [\n        "fotones",\n        "potencial_efectivo",\n        "solve",\n        "diff"\n      ]\n    },\n    {\n      "nivel": 9,\n      "id": "isco_schwarzschild",\n      "titulo": "ISCO: r = 6GM/c²",\n      "enunciado": "Con el potencial efectivo V(r) = (1 - 2*M/r)*(1 + L**2/r**2) (G = c = 1), las órbitas circulares cumplen dV/dr = 0 y son estables si d2V/dr2 > 0. Demuestra que, tras imponer L**2 = M*r**2/(r - 3*M), la condición de estabilidad marginal d2V/dr2 = 0 se cumple en r = 6*M (en unidades con constantes: r = 6*G*M/c**2) y que para r > 6*M se tiene d2V/dr2 > 0. Criterio: d2V/dr2, evaluada con ese L**2 en r = 6*M, se simplifica a 0 y en r = 7*M da un valor positivo para M > 0.",\n      "idea": "Calcula `diff(V, r, 2)`, sustituye L**2 de la órbita circular y resuelve la ecuación resultante con `solve` o factoriza con `factor`.",\n      "etiquetas": [\n        "isco",\n        "potencial_efectivo",\n        "diff",\n        "solve",\n        "estabilidad"\n      ]\n    },\n    {\n      "nivel": 10,\n      "id": "temperatura_de_hawking",\n      "titulo": "Temperatura de Hawking como relación",\n      "enunciado": "Se toma como dato la temperatura de Hawking de un agujero negro de Schwarzschild de masa M: T = hbar*c**3/(8*pi*G*M*k_B). Demuestra simbólicamente que T es inversamente proporcional a M (T*M es constante, su derivada respecto de M es 0) y calcula numéricamente T para una masa solar (M = 1.989e30 kg, hbar = 1.0546e-34 J*s, c = 2.998e8 m/s, G = 6.674e-11 SI, k_B = 1.381e-23 J/K), comprobando que está entre 5e-8 K y 7e-8 K. Criterio: diff(T*M, M) se simplifica a 0 y el valor numérico cae en el intervalo.",\n      "idea": "Define T con símbolos positivos, comprueba la derivada de T*M con `diff` y sustituye los valores numéricos con `subs`/`evalf`.",\n      "etiquetas": [\n        "hawking",\n        "termodinamica",\n        "numerico",\n        "diff"\n      ]\n    },\n    {\n      "nivel": 11,\n      "id": "entropia_bekenstein_hawking",\n      "titulo": "Entropía de Bekenstein-Hawking y primer principio",\n      "enunciado": "Sea r_s = 2*G*M/c**2, el área del horizonte A = 4*pi*r_s**2 y la entropía S = k_B*c**3*A/(4*G*hbar). Demuestra simbólicamente que S = 4*pi*k_B*G*M**2/(hbar*c) y que S es proporcional al área (S/A es constante respecto de M). Con T = hbar*c**3/(8*pi*G*M*k_B) y E = M*c**2, comprueba el primer principio dE = T*dS, es decir, que dS/dM = c**2/T. Criterio: simplify(S - 4*pi*k_B*G*M**2/(hbar*c)) = 0, diff(S/A, M) = 0 y simplify(diff(S, M) - c**2/T) = 0.",\n      "idea": "Sustituye r_s en A y S, simplifica, deriva S respecto de M con `diff` y compara con c**2/T usando `simplify`.",\n      "etiquetas": [\n        "entropia",\n        "bekenstein_hawking",\n        "termodinamica",\n        "diff",\n        "simplify"\n      ]\n    },\n    {\n      "nivel": 12,\n      "id": "schwarzschild_vacio_einstein",\n      "titulo": "Schwarzschild resuelve las ecuaciones de vacío",\n      "enunciado": "Con coordenadas (t, r, theta, phi), unidades G = c = 1 y signatura (-,+,+,+), considera la métrica de Schwarzschild ds**2 = -(1 - 2*M/r)*dt**2 + dr**2/(1 - 2*M/r) + r**2*dtheta**2 + r**2*sin(theta)**2*dphi**2 (diagonal, M > 0 constante, r > 2*M). Calcula con SymPy los símbolos de Christoffel, el tensor de Riemann y el tensor de Ricci R_{mu nu} y demuestra que todas sus componentes son idénticamente cero (ecuaciones de vacío de Einstein R_{mu nu} = 0). Como comprobación de contraste, el escalar de Kretschmann debe valer 48*M**2/r**6 (distinto de cero: la singularidad es real). Criterio: cada componente de Ricci se simplifica a 0 y Kretschmann - 48*M**2/r**6 se simplifica a 0.",\n      "idea": "Implementa Christoffel con la fórmula Gamma^a_{bc} = (1/2) g^{ad}(d_b g_{dc} + d_c g_{db} - d_d g_{bc}), construye Riemann y Ricci por contracción con `diff` y `simplify` componente a componente, y contrasta con Kretschmann.",\n      "etiquetas": [\n        "relatividad_general",\n        "ricci",\n        "christoffel",\n        "tensores",\n        "schwarzschild",\n        "diff",\n        "simplify"\n      ]\n    }\n  ],\n  "abiertos": [\n    {\n      "id": "conjetura_de_goldbach",\n      "titulo": "Conjetura de Goldbach",\n      "enunciado": "Todo número par mayor que 2 es la suma de dos números primos.",\n      "backend": "lean",\n      "nota": "Versión acotada verificable (sin Mathlib): definir `esPrimo (p : Nat) : Bool` por divisores menores que p y comprobar con `decide` (o `native_decide` si se permite) que todo par n con 4 ≤ n ≤ 1000 es suma de dos primos. El enunciado general para todos los pares sigue ABIERTO; no se afirma que el sistema pueda resolverlo."\n    },\n    {\n      "id": "conjetura_de_collatz",\n      "titulo": "Conjetura de Collatz (3n+1)",\n      "enunciado": "Partiendo de cualquier entero positivo n y aplicando repetidamente n -> n/2 si n es par y n -> 3n+1 si es impar, se llega siempre a 1.",\n      "backend": "lean",\n      "nota": "Versión acotada verificable: definir la función de pasos con combustible (fuel) de forma recursiva y demostrar por `decide` que para todo n con 1 ≤ n ≤ 200 se alcanza 1 en a lo sumo, p. ej., 150 pasos (el máximo para n ≤ 200 es 124, alcanzado por n = 171). El enunciado general sigue ABIERTO; no se afirma que el sistema pueda resolverlo."\n    },\n    {\n      "id": "primos_gemelos",\n      "titulo": "Conjetura de los primos gemelos",\n      "enunciado": "Existen infinitos pares de primos (p, p+2).",\n      "backend": "lean",\n      "nota": "Versión acotada verificable: con `esPrimo` definido por divisores, demostrar por `decide` que para cada cota N en un rango pequeño (por ejemplo N = 10, 100, 1000) existe un par de gemelos (p, p+2) con p > N hasta un límite explícito, o exhibir testigos concretos como (3,5), (11,13), (101,103). La afirmación de infinitud sigue ABIERTA; no se afirma que el sistema pueda resolverla."\n    },\n    {\n      "id": "numero_perfecto_impar",\n      "titulo": "No existen números perfectos impares",\n      "enunciado": "Ningún número impar es igual a la suma de sus divisores propios positivos (conjetura abierta: no se conoce ningún número perfecto impar).",\n      "backend": "lean",\n      "nota": "Versión acotada verificable: definir recursivamente la suma de divisores propios `sigmaPropia n` y comprobar por `decide` que ningún impar n con 1 ≤ n ≤ 2000 cumple sigmaPropia n = n (y que los pares 6 y 28 sí son perfectos). La no existencia general sigue ABIERTA (hay cotas computacionales enormes, pero no demostración); no se afirma que el sistema pueda resolverla."\n    },\n    {\n      "id": "conjetura_de_legendre",\n      "titulo": "Conjetura de Legendre",\n      "enunciado": "Para todo entero n ≥ 1 existe un primo p con n**2 < p < (n+1)**2.",\n      "backend": "lean",\n      "nota": "Versión acotada verificable: con `esPrimo` por divisores, comprobar por `decide` que para todo n con 1 ≤ n ≤ 100 existe un primo en el intervalo (n*n, (n+1)*(n+1)). El enunciado general sigue ABIERTO; no se afirma que el sistema pueda resolverlo."\n    },\n    {\n      "id": "erdos_straus",\n      "titulo": "Conjetura de Erdős–Straus",\n      "enunciado": "Para todo entero n ≥ 2 existen enteros positivos x, y, z con 4/n = 1/x + 1/y + 1/z; sin fracciones: 4*x*y*z = n*(x*y + y*z + z*x).",\n      "backend": "lean",\n      "nota": "Versión acotada verificable: para cada n con 2 ≤ n ≤ 100 buscar con `decide` (o aportando testigos explícitos por `exact ⟨x, y, z, ...⟩`) una terna que cumpla 4*x*y*z = n*(x*y + y*z + z*x), con x, y, z acotados por n**2 o similar. La conjetura general (todo n) sigue ABIERTA, aunque está comprobada por ordenador hasta cotas muy grandes; no se afirma que el sistema pueda resolverla."\n    }\n  ]\n}\n')
print('curriculo/curriculo.json escrito')

In [ ]:
!python demostrador.py curriculo --pista lean --modelo {MODELO} --lean {LEAN} --dir {DIR}/curriculo --horas 3 --directos 4 --replanes 2

Física (SymPy), del nivel 1 al 12. Cámbialo por `--desde 5 --hasta 9` para ir por tramos.

In [ ]:
!python demostrador.py curriculo --pista sympy --modelo {MODELO} --dir {DIR}/curriculo --horas 3

Mira la biblioteca acumulada (lo que el sistema ya sabe demostrar) y el progreso:

In [ ]:
import json, os
for pista in ('lean', 'sympy'):
    f = f'{DIR}/curriculo/progreso_{pista}.json'
    if os.path.exists(f):
        print(pista, json.load(open(f)))